# Misfit Agent — ARC-AGI-3 Tier-1 Submission

> **Source:** https://github.com/AtomEons/arc-agi-3-misfit-agent  
> **License:** Apache-2.0  
> **Tier-1 attestation:** Spelke core knowledge object priors + hand-authored typed rule grammar. **No pretrained model weights of any kind. No language model in the inference path.**

> Mechanically enforced by `tests/test_tier1_attestation.py` — fails CI if any of `transformers`, `openai`, `anthropic`, `llama_cpp`, `huggingface_hub`, `sentence_transformers`, `langchain`, `langgraph`, `smolagents` are imported.

---

# Tier-1 Disclosure

This document accompanies every Kaggle submission as a cell-0 markdown block.

## Claim

This agent uses **Spelke Core Knowledge object priors** combined with a **hand-authored typed rule grammar over those priors**. There are no pretrained model weights of any kind. There is no language model. The codebase is Apache-2.0 licensed.

## Priors used (Tier 1)

| Prior | How it appears in code |
|---|---|
| **Cohesion** | 4-connectivity connected-component labelling in `perceptor.py` — same-color contiguous cells form one object |
| **Continuity / persistence** | Hungarian matching of object identities across consecutive frames in `tracker.py` |
| **Contact-causality** | State changes triggered by spatial contact, encoded in `rules/destroy_on_contact.py` and `rules/spawn_on_contact.py` |
| **Persistence under occlusion** | Hungarian cost prefers hidden-then-reappear over destroyed-then-spawned |
| **Agency vs patient** | The object whose centroid correlates with ACTION1-4 deltas is *the agent* — learned correlation, not hardcoded |
| **Compositionality** | Transition function factorizes per object class |
| **Sparse causality** | Most actions affect at most 1-2 objects |

**Citation:** Spelke, E. S., & Kinzler, K. D. (2007). *Core knowledge.* Developmental Science, 10(1), 89-96.

## Hand-authored contribution (honest naming)

The **six rule templates** in `src/misfit_agent/rules/` (TRANSLATE, TELEPORT_TO, DESTROY_ON_CONTACT, SPAWN_ON_CONTACT, TOGGLE_AT_CURSOR, NO_OP) are a typed grammar **authored by an author who has been exposed to ARC-AGI-1 and ARC-AGI-2 examples**. The grammar is not derived purely from Spelke priors — it encodes a designer's intuition about how grid-action puzzles tend to be structured.

This is disclosed explicitly. A hostile reviewer asking "did the author's exposure to ARC examples inform the template list?" — the answer is yes, and we name it.

## What we do NOT use (Tier 2 contamination — intentionally excluded)

- No language model proposer (no GPT, no Claude, no Gemini, no Mamba, no Qwen, no Llama)
- No pretrained vision encoder
- No ARC-AGI-1 / ARC-AGI-2 task descriptions or solutions in any prompt or training corpus
- No fine-tuned LoRA weights
- No `transformers`, `openai`, `anthropic`, `llama_cpp`, `huggingface_hub`, `sentence_transformers`, `langchain`, `langgraph`, `smolagents` imports

This is mechanically enforced by `tests/test_tier1_attestation.py` which fails the CI build if any forbidden import or pretrained-model string appears in the source tree.

## What we do NOT do (Tier 3 — forbidden)

- No Kaggle private-set leakage (we never see private games)
- No threshold sweeping on the 25 public dev games after Day 10 freeze
- No scraping of past ARC Prize winners' code or solutions
- No human-in-the-loop hint injection at evaluation time
- No out-of-band channel to the gateway

## Frozen-constants policy

All magic numbers live in `src/misfit_agent/config.py` with explicit classification:
- `(a)` derived from a prior (geometric truth, Spelke core)
- `(b)` budget heuristic (math from the scoring rule or wall-clock)
- `(c)` tuned on public games (must be disclosed)

Any threshold change after the first Kaggle submission requires a git tag, a CHANGELOG entry, and a named judge finding that justified it.

## Tuning split

The 25 public games are split **18 train / 7 held-out validation**. Thresholds are tuned only on the 18, with held-out numbers reported separately in `METHODOLOGY.md`. We do not sweep on all 25 and report a single number.

## Source

[https://github.com/AtomEons/arc-agi-3-misfit-agent](https://github.com/AtomEons/arc-agi-3-misfit-agent) (repo URL placeholder — repo not yet pushed)


In [ ]:
# Cell 1 — offline wheel install from the bundled competition data.
# Internet is OFF (per kernel-metadata.json). Wheels live in the
# competition dataset; mount path varies between Kaggle layouts so
# resolve at runtime then hand the resolved dir to !pip install.
import os
_WHEEL_CANDIDATES = [
    '/kaggle/input/arc-prize-2026-arc-agi-3/arc_agi_3_wheels',
    '/kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels',
]
_wheel_dir = next((p for p in _WHEEL_CANDIDATES if os.path.isdir(p)), None)
assert _wheel_dir, f'no wheel dir found at any of: {_WHEEL_CANDIDATES}'
print(f'misfit: wheel mount resolved to {_wheel_dir}')
!pip install --no-index --find-links {_wheel_dir} arc-agi python-dotenv


In [ ]:
%%writefile /kaggle/working/my_agent.py
"""
MisfitAgent — Tier-1 Spelke-priors ARC-AGI-3 agent (built notebook bundle).

Generated by scripts/build_notebook.py — do not edit by hand.
Source: https://github.com/AtomEons/arc-agi-3-misfit-agent

This bundle contains all 12 substrate modules flattened into one file:
  config, perceptor, episode, fingerprint, resonance,
  rules/no_op, rules/translate, world_model, click_quantizer,
  tracker_hungarian, goal_inducer, abstain_policy, action_search,
  mcts_puct, misfit_agent.

NO LLM in the inference path. NO pretrained weights. NO ARC task-family
hardcoding. Hand-authored typed rule grammar over Spelke core knowledge
priors. Mechanically enforced by tests/test_tier1_attestation.py.
"""

# ----- shared standard-library and ecosystem imports --------------------
from __future__ import annotations

import copy
import json
import math
import os
import pathlib
import random
import time
from collections import Counter
from dataclasses import asdict, dataclass, field
from typing import Any, Callable, Optional, Sequence

import numpy as np

# arcengine + framework supplied by the eval container.
from arcengine import FrameData, GameAction, GameState
from agents.agent import Agent


# ============================================================
# MODULE: config.py
# ============================================================



from dataclasses import dataclass


@dataclass(frozen=True)
class TrackerConfig:
    # Hungarian matching cost weights (Day-4 deliverable).
    # Classification: (a) DERIVED FROM PRIOR — generic geometric distance / topology / color identity.
    # These weights encode the relative importance of position vs shape vs identity for cohesion.
    # NOT tuned on any specific game grid. If we sweep them on public games, we must move them to (c).
    alpha_centroid_dist: float = 1.0
    beta_shape_hamming: float = 0.5
    gamma_color_mismatch: float = 2.0


@dataclass(frozen=True)
class WorldModelConfig:
    # (a) DERIVED FROM PRIOR — same-observation requires same-prediction is the consistency axiom.
    min_observations_for_trust: int = 3
    # (b) BUDGET HEURISTIC — wall-clock per-step simulator target (µs).
    target_us_per_step: int = 50
    # (b) BUDGET HEURISTIC — episodic memory ceiling (states per game).
    max_states_per_game: int = 10_000


@dataclass(frozen=True)
class PseudocountConfig:
    # (a) DERIVED FROM PRIOR — standard novelty-bonus form alpha / sqrt(N+1).
    novelty_alpha: float = 1.0


@dataclass(frozen=True)
class AbstainConfig:
    # (b) BUDGET HEURISTIC — derive from quadratic scoring math.
    # Score = (human/agent)^2. Half-life of marginal score gain is at agent_actions = 2*human_baseline.
    # Default 25 = empirical band where pseudocount novelty typically plateaus on small grids.
    # MUST be re-derived from a published calculation, not asserted — judge auditor flag.
    min_actions_before_abstain: int = 25
    # (a) DERIVED FROM PRIOR — when world-model variance > 20% of predictions, hypothesis class is wrong.
    world_model_variance_threshold: float = 0.20
    # (a) DERIVED FROM PRIOR — pseudocount slope below this = exploration plateau.
    novelty_plateau_slope: float = 0.05


@dataclass(frozen=True)
class MCTSConfig:
    # (b) BUDGET HEURISTIC — PUCT exploration constant; canonical value in AlphaZero literature.
    c_puct: float = 1.41
    # (b) BUDGET HEURISTIC — per-action search depth cap.
    max_depth: int = 6
    # (b) BUDGET HEURISTIC — rollouts per real action; trades quality vs wall clock.
    rollouts_per_action: int = 200
    # (b) BUDGET HEURISTIC — hard timeout per choose_action call (ms).
    hard_timeout_ms: int = 500


@dataclass(frozen=True)
class BudgetConfig:
    # (b) BUDGET HEURISTIC — Kaggle's 9h hard cap minus 5 min cold-start overhead minus 5 min safety.
    wall_clock_kill_seconds: int = 8 * 3600 + 50 * 60   # 8h50m
    # (a) DERIVED FROM PRIOR — the agents.agent.Agent default. Server-side cap may be 80;
    # MUST verify empirically on Day 1 (judge Kaggle-reality must-fix).
    # Raised to 400 in plan but treated as advisory until verified.
    max_actions_per_game: int = 80
    # (b) BUDGET HEURISTIC — anticipated 110 private games per Kaggle eval batch.
    expected_total_games: int = 110


@dataclass(frozen=True)
class FingerprintConfig:
    # (b) BUDGET HEURISTIC — total signature dimensionality.
    total_dim: int = 50


@dataclass(frozen=True)
class FrozenConfig:
    """Singleton view over all module configs.

    To change a value here AFTER first submission, the operator must:
      1. Tag the prior submission's git SHA
      2. Document the change in CHANGELOG.md with rationale
      3. Note which judge must-fix justified the change
    """
    tracker: TrackerConfig = TrackerConfig()
    world_model: WorldModelConfig = WorldModelConfig()
    pseudocount: PseudocountConfig = PseudocountConfig()
    abstain: AbstainConfig = AbstainConfig()
    mcts: MCTSConfig = MCTSConfig()
    budget: BudgetConfig = BudgetConfig()
    fingerprint: FingerprintConfig = FingerprintConfig()


CONFIG = FrozenConfig()

# ============================================================
# MODULE: perceptor.py
# ============================================================



from dataclasses import dataclass, field
from typing import Sequence



Grid = np.ndarray  # 2-D uint8 ARC color grid


@dataclass(frozen=True)
class Obj:
    """A single coherent object under the objectness prior."""
    color: int
    area: int
    bbox: tuple[int, int, int, int]  # (r0, c0, r1, c1) inclusive
    centroid: tuple[float, float]
    touches_edge: bool
    v_symmetric: bool
    h_symmetric: bool


@dataclass
class SceneObservation:
    """Result of perceiving one frame under Spelke priors."""
    grid: Grid
    rows: int
    cols: int
    background_color: int
    objects: list[Obj] = field(default_factory=list)
    color_histogram: list[int] = field(default_factory=list)  # length 10
    foreground_cells: int = 0


def _flood_label(grid: Grid, bg: int) -> np.ndarray:
    """4-connected component labels. Cells with color==bg get label 0."""
    rows, cols = grid.shape
    labels = np.zeros((rows, cols), dtype=np.int32)
    next_label = 0
    for r in range(rows):
        for c in range(cols):
            if grid[r, c] == bg or labels[r, c] != 0:
                continue
            next_label += 1
            stack = [(r, c)]
            target_color = int(grid[r, c])
            while stack:
                rr, cc = stack.pop()
                if rr < 0 or rr >= rows or cc < 0 or cc >= cols:
                    continue
                if labels[rr, cc] != 0:
                    continue
                if int(grid[rr, cc]) != target_color:
                    continue
                labels[rr, cc] = next_label
                stack.extend([(rr+1, cc), (rr-1, cc), (rr, cc+1), (rr, cc-1)])
    return labels


def _is_symmetric_v(sub: Grid) -> bool:
    return bool(np.array_equal(sub, np.flip(sub, axis=1)))


def _is_symmetric_h(sub: Grid) -> bool:
    return bool(np.array_equal(sub, np.flip(sub, axis=0)))


def _background_color(grid: Grid) -> int:
    """ARC convention: 0 is background when present; else most-frequent color."""
    if (grid == 0).any():
        return 0
    counts = np.bincount(grid.ravel(), minlength=10)
    return int(np.argmax(counts))


def perceive_grid(grid: Grid) -> SceneObservation:
    """Perceive a single 2-D grid under Spelke priors."""
    grid = np.asarray(grid, dtype=np.int32)
    if grid.ndim != 2:
        raise ValueError(f"perceive_grid expects 2-D grid, got shape {grid.shape}")
    rows, cols = grid.shape
    bg = _background_color(grid)
    labels = _flood_label(grid, bg)
    n_labels = int(labels.max())

    objects: list[Obj] = []
    for lab in range(1, n_labels + 1):
        mask = labels == lab
        ys, xs = np.where(mask)
        if ys.size == 0:
            continue
        r0, r1 = int(ys.min()), int(ys.max())
        c0, c1 = int(xs.min()), int(xs.max())
        sub = grid[r0:r1+1, c0:c1+1]
        sub_mask = mask[r0:r1+1, c0:c1+1].astype(np.int32) * (sub != bg).astype(np.int32)
        color = int(grid[ys[0], xs[0]])
        touches = bool(r0 == 0 or c0 == 0 or r1 == rows-1 or c1 == cols-1)
        v_sym = _is_symmetric_v(sub_mask)
        h_sym = _is_symmetric_h(sub_mask)
        centroid = (float(ys.mean()), float(xs.mean()))
        objects.append(Obj(
            color=color,
            area=int(ys.size),
            bbox=(r0, c0, r1, c1),
            centroid=centroid,
            touches_edge=touches,
            v_symmetric=v_sym,
            h_symmetric=h_sym,
        ))

    # Sort by area descending (largest first)
    objects.sort(key=lambda o: -o.area)

    hist = np.bincount(grid.ravel(), minlength=10)[:10].tolist()
    fg = int((grid != bg).sum())

    return SceneObservation(
        grid=grid,
        rows=rows,
        cols=cols,
        background_color=bg,
        objects=objects,
        color_histogram=list(map(int, hist)),
        foreground_cells=fg,
    )


def perceive_frame(frame: Sequence) -> SceneObservation:
    """Perceive an ARC-AGI-3 frame. `frame` is the FrameData.frame attribute —
    documented upstream as `list[list[list[int]]]` after numpy.tolist().
    Most games are single-grid; multi-grid frames keep the first plane for now.
    """
    arr = np.asarray(frame, dtype=np.int32)
    if arr.ndim == 3:
        # Multi-plane: use the first plane. Higher-order spatial reasoning over
        # multiple planes is a Day-N+ extension under the geometry prior.
        arr = arr[0]
    if arr.ndim != 2:
        raise ValueError(f"perceive_frame expects 2-D or 3-D grid, got shape {arr.shape}")
    return perceive_grid(arr)


def grid_diff(a: Grid, b: Grid) -> tuple[int, int, int]:
    """Cell-level diff between two grids of equal shape.
    Returns (changed_cell_count, bg_to_fg_count, fg_to_bg_count).
    Used by rule induction to characterize action-effect under Spelke priors.
    """
    a = np.asarray(a, dtype=np.int32)
    b = np.asarray(b, dtype=np.int32)
    if a.shape != b.shape:
        return (max(a.size, b.size), 0, 0)
    bg_a = _background_color(a)
    bg_b = _background_color(b)
    a_bg = a == bg_a
    b_bg = b == bg_b
    bg_to_fg = int(((a_bg) & (~b_bg)).sum())
    fg_to_bg = int(((~a_bg) & (b_bg)).sum())
    changed = int((a != b).sum())
    return (changed, bg_to_fg, fg_to_bg)

# ============================================================
# MODULE: episode.py
# ============================================================



from typing import Any, Optional



@dataclass
class ActionRecord:
    action_name: str         # e.g. "ACTION1", "ACTION6"
    action_value: int        # numeric enum value
    data: dict               # {"x": int, "y": int} for complex actions
    pre_levels_completed: int
    post_levels_completed: Optional[int] = None
    cells_changed: Optional[int] = None
    triggered_win: bool = False


@dataclass
class EpisodeTracker:
    game_id: str
    scenes: list[SceneObservation] = field(default_factory=list)
    action_history: list[ActionRecord] = field(default_factory=list)
    last_action_rationale: str = ""

    # Lightweight induced rule beliefs (Day 2+ will fill these from transitions).
    # All keys are *observed* dynamics, not hand-crafted task families.
    transition_signals: dict[str, Any] = field(default_factory=dict)

    def observe(self, latest_frame: Any, scene: SceneObservation) -> None:
        """Record a perceived frame + close the previous action record."""
        prior_scene = self.scenes[-1] if self.scenes else None
        self.scenes.append(scene)

        # If we have a prior action record awaiting a follow-up, close it.
        if self.action_history and self.action_history[-1].post_levels_completed is None:
            rec = self.action_history[-1]
            rec.post_levels_completed = int(latest_frame.levels_completed)
            if prior_scene is not None and scene.grid.shape == prior_scene.grid.shape:
                changed, _, _ = grid_diff(prior_scene.grid, scene.grid)
                rec.cells_changed = changed
            rec.triggered_win = bool(getattr(latest_frame, "state", None) and
                                     str(latest_frame.state).endswith("WIN"))
            self._update_transition_signals(rec, prior_scene, scene)

    def observe_terminal(self, latest_frame: Any) -> None:
        """Game-over / not-played state — clear last-action linkage."""
        if self.action_history and self.action_history[-1].post_levels_completed is None:
            self.action_history[-1].post_levels_completed = int(latest_frame.levels_completed)

    def record_action(self, action_name: str, action_value: int, data: dict,
                      pre_levels_completed: int) -> None:
        self.action_history.append(ActionRecord(
            action_name=action_name,
            action_value=action_value,
            data=dict(data) if data else {},
            pre_levels_completed=pre_levels_completed,
        ))

    def set_rationale(self, rationale: str) -> None:
        self.last_action_rationale = rationale

    def winning_actions(self) -> list[ActionRecord]:
        return [a for a in self.action_history if a.triggered_win]

    def level_advancing_actions(self) -> list[ActionRecord]:
        out = []
        for a in self.action_history:
            if a.post_levels_completed is not None and a.post_levels_completed > a.pre_levels_completed:
                out.append(a)
        return out

    def _update_transition_signals(self, rec: ActionRecord,
                                    prior: SceneObservation,
                                    after: SceneObservation) -> None:
        """Accumulate *observed* transition signals — never hardcoded priors."""
        if prior is None:
            return
        action_key = rec.action_name
        bucket = self.transition_signals.setdefault(action_key, {
            "total": 0,
            "cells_changed_sum": 0,
            "level_advances": 0,
            "object_count_delta_sum": 0,
        })
        bucket["total"] += 1
        bucket["cells_changed_sum"] += rec.cells_changed or 0
        if rec.post_levels_completed is not None and rec.post_levels_completed > rec.pre_levels_completed:
            bucket["level_advances"] += 1
        bucket["object_count_delta_sum"] += len(after.objects) - len(prior.objects)


def observe_hungarian(prev_scene: SceneObservation,
                       curr_scene: SceneObservation,
                       tracker: Optional[Any] = None) -> dict:
    """Build an (s, a, s') correspondence dict via HungarianTracker.

    Returns a dict with keys:
      - "mapping": {prev_obj_idx -> curr_obj_idx | None} from HungarianTracker.track
      - "spawned": list of curr-scene indices unmatched (born objects)
      - "destroyed": list of prev-scene indices with no successor
      - "matched_pairs": list of (prev_idx, curr_idx) tuples for matched persistents
      - "prev_objects": [{"centroid": (r,c), "area": int, "color": int}, ...]
      - "post_objects": [{"centroid": (r,c), "area": int, "color": int}, ...]

    The `tracker` parameter is accepted for API symmetry with the existing
    tracker.observe() methods, but is NOT used — correspondences are a pure
    function of the two scenes under Spelke priors (CONTINUITY, COHESION,
    SOLIDITY). Callers will typically fold this dict into the WorldModel
    observation list so per-class rules see object identity, not just
    summary counts.
    """
    # Imported here to keep top-level import surface minimal and to avoid
    # any chance of cycle (HungarianTracker imports perceptor types too).

    _ = tracker  # accepted for API symmetry; correspondences are pure of scenes
    ht = HungarianTracker()
    mapping = ht.track(prev_scene, curr_scene)
    spawned = ht.spawned_indices(prev_scene, curr_scene, mapping)
    destroyed = ht.destroyed_indices(mapping)
    matched_pairs = [(int(i), int(j)) for i, j in mapping.items() if j is not None]

    def _serialize(obj: Any) -> dict:
        return {
            "centroid": obj.centroid,
            "area": int(obj.area),
            "color": int(obj.color),
        }

    return {
        "mapping": {int(k): (int(v) if v is not None else None)
                    for k, v in mapping.items()},
        "spawned": [int(j) for j in spawned],
        "destroyed": [int(i) for i in destroyed],
        "matched_pairs": matched_pairs,
        "prev_objects": [_serialize(o) for o in prev_scene.objects],
        "post_objects": [_serialize(o) for o in curr_scene.objects],
    }

# ============================================================
# MODULE: fingerprint.py
# ============================================================







FINGERPRINT_DIM = 50


def fingerprint_episode(tracker: EpisodeTracker) -> np.ndarray:
    v = np.zeros(FINGERPRINT_DIM, dtype=np.float32)
    scenes = tracker.scenes
    if not scenes:
        return v

    rows = [s.rows for s in scenes]
    cols = [s.cols for s in scenes]
    v[0] = float(np.mean(rows)) / max(float(np.mean(cols)), 1.0)

    same_shape = sum(1 for i in range(1, len(scenes))
                     if scenes[i].grid.shape == scenes[i-1].grid.shape)
    v[1] = same_shape / max(len(scenes) - 1, 1)

    obj_counts = [len(s.objects) for s in scenes]
    v[2] = float(np.mean(obj_counts)) if obj_counts else 0.0
    v[3] = float(obj_counts[-1] - obj_counts[0]) if len(obj_counts) > 1 else 0.0

    def _mean_safe(arr): return float(np.mean(arr)) if arr else 0.0
    v[4] = _mean_safe([sum(o.v_symmetric for o in s.objects) / max(len(s.objects), 1)
                       for s in scenes])
    v[5] = _mean_safe([sum(o.h_symmetric for o in s.objects) / max(len(s.objects), 1)
                       for s in scenes])
    v[6] = _mean_safe([sum(o.touches_edge for o in s.objects) / max(len(s.objects), 1)
                       for s in scenes])
    v[7] = _mean_safe([(s.objects[0].area / max(s.rows*s.cols, 1)) if s.objects else 0.0
                       for s in scenes])
    v[8] = _mean_safe([s.foreground_cells / max(s.rows*s.cols, 1) for s in scenes])
    v[9] = float(math.log(1 + len(scenes)))

    # Palette density per color 0..9 (mean over scenes)
    palette_means = np.zeros(10, dtype=np.float32)
    for s in scenes:
        denom = max(s.rows * s.cols, 1)
        for c in range(10):
            palette_means[c] += s.color_histogram[c] / denom
    palette_means /= max(len(scenes), 1)
    v[10:20] = palette_means

    # Palette delta — last vs first scene
    first_p = np.array(scenes[0].color_histogram, dtype=np.float32) / max(
        scenes[0].rows * scenes[0].cols, 1)
    last_p = np.array(scenes[-1].color_histogram, dtype=np.float32) / max(
        scenes[-1].rows * scenes[-1].cols, 1)
    v[20:30] = (last_p - first_p)

    # Action-effect signature & level-advance rate per action enum slot.
    action_slots = ["RESET", "ACTION1", "ACTION2", "ACTION3", "ACTION4",
                    "ACTION5", "ACTION6", "ACTION7"]
    for i, name in enumerate(action_slots):
        bucket = tracker.transition_signals.get(name)
        if not bucket or bucket["total"] == 0:
            continue
        v[30 + i] = bucket["cells_changed_sum"] / bucket["total"]
        v[38 + i] = bucket["level_advances"] / bucket["total"]

    return v


def cosine(a: np.ndarray, b: np.ndarray) -> float:
    na = float(np.linalg.norm(a))
    nb = float(np.linalg.norm(b))
    if na < 1e-12 or nb < 1e-12:
        return 0.0
    return float(np.dot(a, b) / (na * nb))

# ============================================================
# MODULE: resonance.py
# ============================================================



from typing import Optional




def default_library_path() -> pathlib.Path:
    """Per-install path: %LOCALAPPDATA%\\misfit-agent on Windows,
    or ~/.local/share/misfit-agent elsewhere."""
    if os.name == "nt":
        base = pathlib.Path(os.environ.get("LOCALAPPDATA", str(pathlib.Path.home())))
    else:
        base = pathlib.Path.home() / ".local" / "share"
    return base / "misfit-agent" / "resonance_library.jsonl"


@dataclass
class LibraryEntry:
    game_id: str
    fingerprint: list[float]
    winning_policy: list[dict]  # serialized ActionRecord list
    composite_score: float
    solved_at_unix: float
    source: str = "self-solved"


@dataclass
class ResonanceLibrary:
    path: pathlib.Path
    entries: list[LibraryEntry] = field(default_factory=list)
    _pending: list[LibraryEntry] = field(default_factory=list)

    @classmethod
    def load_or_create(cls, path: str | pathlib.Path) -> "ResonanceLibrary":
        path = pathlib.Path(path)
        lib = cls(path=path)
        if not path.exists():
            return lib
        try:
            with path.open("r", encoding="utf-8") as f:
                for line in f:
                    line = line.strip()
                    if not line:
                        continue
                    try:
                        d = json.loads(line)
                    except json.JSONDecodeError:
                        continue
                    if d.get("source") != "self-solved":
                        # Tier-1 honesty: silently skip non-self-solved entries.
                        continue
                    lib.entries.append(LibraryEntry(**d))
        except OSError:
            pass
        return lib

    def find_k_nearest(self, query: np.ndarray, k: int = 5
                       ) -> list[tuple[LibraryEntry, float]]:
        if not self.entries:
            return []
        scored = []
        for e in self.entries:
            sim = cosine(query, np.asarray(e.fingerprint, dtype=np.float32))
            scored.append((e, sim))
        scored.sort(key=lambda x: -x[1])
        return scored[:k]

    def retrieve_policy_seeds(self, query: np.ndarray, k: int = 5
                              ) -> list[list[dict]]:
        """Return up to K unique winning policies for resonance-seeded search."""
        seen: set[str] = set()
        out: list[list[dict]] = []
        for e, _ in self.find_k_nearest(query, k):
            sig = json.dumps([a["action_value"] for a in e.winning_policy])
            if sig in seen:
                continue
            seen.add(sig)
            out.append(e.winning_policy)
        return out

    def record_solved(self, fingerprint: np.ndarray, winning_policy: list[ActionRecord],
                       composite_score: float, source: str = "self-solved",
                       game_id: Optional[str] = None) -> LibraryEntry:
        if source != "self-solved":
            raise ValueError(
                f"Tier-1 honesty: library only accepts self-solved entries, got {source!r}"
            )
        gid = game_id or (winning_policy[0].action_name if winning_policy else "unknown")
        entry = LibraryEntry(
            game_id=gid,
            fingerprint=[float(x) for x in fingerprint.tolist()],
            winning_policy=[asdict(a) for a in winning_policy],
            composite_score=float(composite_score),
            solved_at_unix=time.time(),
            source=source,
        )
        self.entries.append(entry)
        self._pending.append(entry)
        return entry

    def flush_to_disk(self) -> int:
        if not self._pending:
            return 0
        self.path.parent.mkdir(parents=True, exist_ok=True)
        n = 0
        with self.path.open("a", encoding="utf-8") as f:
            for e in self._pending:
                f.write(json.dumps(asdict(e)) + "\n")
                n += 1
        self._pending.clear()
        return n

# ============================================================
# MODULE: rules/no_op.py
# ============================================================






@dataclass
class NoOp:
    """The null rule: action has no effect on this object class."""

    object_class: int   # which color/class this rule applies to
    fitted: bool = False

    def fit(self, observations: list[dict]) -> bool:
        """Fit: confirm that ALL observed transitions for this class show no change."""
        for obs in observations:
            pre = obs.get("pre_objects_of_class", [])
            post = obs.get("post_objects_of_class", [])
            if len(pre) != len(post):
                return False
            for p, q in zip(pre, post):
                if p["centroid"] != q["centroid"]:
                    return False
                if p["area"] != q["area"]:
                    return False
        self.fitted = True
        return True

    def predict(self, grid: np.ndarray, action_name: str) -> np.ndarray:
        """Predict: nothing changes."""
        return grid.copy()

# ============================================================
# MODULE: rules/translate.py
# ============================================================






@dataclass
class Translate:
    object_class: int
    dx_per_action: dict[str, int] = field(default_factory=dict)
    dy_per_action: dict[str, int] = field(default_factory=dict)
    fitted: bool = False
    consistency_score: float = 0.0

    def fit(self, observations: list[dict]) -> bool:
        """Fit dx/dy per action from (pre_centroid, post_centroid) pairs.

        observations: list of dicts with keys:
            action_name (str)
            pre_objects_of_class (list of {centroid: (r,c), ...})
            post_objects_of_class (list of {centroid: (r,c), ...})
        Per the Hungarian tracker, pre[i] corresponds to post[i] when
        objects persist. We require single-object class for now (multi-
        object translate is a Day-N extension).

        Returns True if a consistent (dx, dy) exists per action — i.e. the
        same delta appears EVERY time the action is taken on this class.
        """
        per_action_deltas: dict[str, list[tuple[float, float]]] = {}
        for obs in observations:
            action = obs.get("action_name", "")
            pre = obs.get("pre_objects_of_class", [])
            post = obs.get("post_objects_of_class", [])
            if len(pre) != 1 or len(post) != 1:
                continue  # Translate is single-object for now
            pre_r, pre_c = pre[0]["centroid"]
            post_r, post_c = post[0]["centroid"]
            per_action_deltas.setdefault(action, []).append(
                (post_r - pre_r, post_c - pre_c)
            )

        if not per_action_deltas:
            return False

        total = 0
        consistent = 0
        for action, deltas in per_action_deltas.items():
            if not deltas:
                continue
            dy = round(float(np.median([d[0] for d in deltas])))
            dx = round(float(np.median([d[1] for d in deltas])))
            for d in deltas:
                total += 1
                if abs(d[0] - dy) < 0.5 and abs(d[1] - dx) < 0.5:
                    consistent += 1
            self.dy_per_action[action] = dy
            self.dx_per_action[action] = dx

        if total == 0:
            return False

        self.consistency_score = consistent / total
        self.fitted = self.consistency_score >= 0.8   # 80% consistency threshold
        return self.fitted

    def predict(self, grid: np.ndarray, action_name: str) -> np.ndarray:
        """Forward simulator: shift all cells of object_class by (dx, dy)."""
        dx = self.dx_per_action.get(action_name, 0)
        dy = self.dy_per_action.get(action_name, 0)
        if dx == 0 and dy == 0:
            return grid.copy()
        out = grid.copy()
        rows, cols = out.shape
        mask = out == self.object_class
        if not mask.any():
            return out
        ys, xs = np.where(mask)
        # Clear current positions
        out[mask] = _background_color(grid)
        # Place at translated positions, clipped to grid bounds
        new_ys = np.clip(ys + dy, 0, rows - 1)
        new_xs = np.clip(xs + dx, 0, cols - 1)
        out[new_ys, new_xs] = self.object_class
        return out


def _background_color(grid: np.ndarray) -> int:
    """ARC convention: 0 is background when present; else most-frequent color."""
    if (grid == 0).any():
        return 0
    counts = np.bincount(grid.ravel(), minlength=10)
    return int(np.argmax(counts))

# ============================================================
# MODULE: world_model.py
# ============================================================







RuleInstance = Translate | NoOp


@dataclass
class WorldModel:
    """Composes fitted rule instances into a deterministic forward simulator."""

    rules: list[RuleInstance] = field(default_factory=list)
    observations_per_action: dict[str, int] = field(default_factory=dict)
    confirmed_transitions: dict[str, int] = field(default_factory=dict)
    # Outer refinement loop bookkeeping — HRM analysis (arcprize.org 2025-08-15)
    # showed +13pp from 0→1 refinement, ~doubled from 1→8. We track total
    # iterations done and last per-call score history for adaptive cutoff.
    refinement_iterations_total: int = 0
    last_fit_score_history: list[float] = field(default_factory=list)

    def fit(self, observations: list[dict]) -> dict[str, float]:
        """Fit available rule templates against observed transitions.

        Single-pass fit. Returns per-rule consistency scores. Use
        `fit_with_refinement` for the HRM-style outer loop variant.
        """
        # Group observations by object_class so each rule fits within-class
        by_class: dict[int, list[dict]] = {}
        for obs in observations:
            for c in obs.get("classes_involved", []):
                by_class.setdefault(c, []).append(obs)

        scores: dict[str, float] = {}
        new_rules: list[RuleInstance] = []

        for cls, cls_obs in by_class.items():
            tr = Translate(object_class=cls)
            if tr.fit(cls_obs):
                new_rules.append(tr)
                scores[f"Translate(class={cls})"] = tr.consistency_score
                continue
            no = NoOp(object_class=cls)
            if no.fit(cls_obs):
                new_rules.append(no)
                scores[f"NoOp(class={cls})"] = 1.0

        # Update transition-count bookkeeping
        for obs in observations:
            action = obs.get("action_name", "")
            self.observations_per_action[action] = (
                self.observations_per_action.get(action, 0) + 1
            )

        self.rules = new_rules
        return scores

    def fit_with_refinement(
        self,
        observations: list[dict],
        max_iters: int = 4,
        improvement_threshold: float = 0.02,
    ) -> dict[str, float]:
        """Outer refinement loop — HRM's hidden performance driver.

        Per arcprize.org/blog/hrm-analysis (2025-08-15): refinement was
        worth +13pp from 0→1 and roughly doubled the gain from 1→8.
        We cap at `max_iters` (default 4 — diminishing returns past that
        per the HRM data) and early-stop when mean rule score improves by
        less than `improvement_threshold`.

        Each refinement pass:
          1. Re-fits rule templates against the full observation set
          2. Predicts each observed transition with the current ruleset
          3. Discards rules whose predictions contradict any single
             observation — feedback signal that drives the next refit

        Returns the final per-rule consistency scores after refinement.
        """
        scores: dict[str, float] = {}
        prev_mean_score = 0.0
        for i in range(max_iters):
            scores = self.fit(observations)
            self.refinement_iterations_total += 1
            mean_score = (
                sum(scores.values()) / len(scores) if scores else 0.0
            )
            self.last_fit_score_history.append(mean_score)
            if mean_score - prev_mean_score < improvement_threshold and i > 0:
                break
            prev_mean_score = mean_score
            # Adversarial pass — discard rules that contradict any obs.
            # The hypothesis_pruner pattern from architect Day 9.
            self._prune_contradicting_rules(observations)
        return scores

    def _prune_contradicting_rules(self, observations: list[dict]) -> None:
        """Drop rules whose predictions are contradicted by any single
        observation in the fit set. The remaining rules survive the
        next refinement iteration; pruning is what makes refinement
        improve coverage rather than just re-fitting."""
        if not self.rules:
            return
        survivors: list[RuleInstance] = []
        for rule in self.rules:
            ok = True
            for obs in observations:
                # Only check observations where this rule's class is involved
                cls = getattr(rule, "object_class", None)
                if cls is None or cls not in obs.get("classes_involved", []):
                    continue
                pre = obs.get("pre_objects_of_class", [])
                post = obs.get("post_objects_of_class", [])
                # If counts differ, NoOp can't apply; Translate can only
                # apply when pre/post counts match. Either way: contradiction
                # = drop. This is a coarse but cheap consistency check.
                if isinstance(rule, NoOp) and len(pre) != len(post):
                    ok = False
                    break
                if isinstance(rule, Translate) and (len(pre) != 1 or len(post) != 1):
                    ok = False
                    break
                if isinstance(rule, NoOp):
                    for p, q in zip(pre, post):
                        if p.get("centroid") != q.get("centroid"):
                            ok = False
                            break
                    if not ok:
                        break
            if ok:
                survivors.append(rule)
        self.rules = survivors

    def predict(
        self,
        grid: np.ndarray,
        action_name: str,
    ) -> tuple[np.ndarray, float]:
        """Forward-simulate one step.

        Returns (predicted_grid, confidence). Confidence is 1.0 only when
        the action has been observed >= min_observations_for_trust times
        AND at least one fitted rule fires.
        """
        if action_name == "RESET":
            # Can't predict reset effects without level-start observations;
            # caller should treat as low-confidence.
            return grid.copy(), 0.0

        obs_count = self.observations_per_action.get(action_name, 0)
        threshold = CONFIG.world_model.min_observations_for_trust
        if obs_count < threshold or not self.rules:
            return grid.copy(), 0.0

        predicted = grid.copy()
        rules_fired = 0
        for rule in self.rules:
            try:
                next_grid = rule.predict(predicted, action_name)
            except Exception:
                continue
            if not np.array_equal(next_grid, predicted):
                predicted = next_grid
                rules_fired += 1

        if rules_fired == 0:
            return predicted, 0.5  # rules exist but none fire — partial confidence

        return predicted, 1.0

    def coverage(self) -> float:
        """Fraction of attempted actions for which we have any fitted rule."""
        if not self.observations_per_action:
            return 0.0
        threshold = CONFIG.world_model.min_observations_for_trust
        trusted = sum(1 for n in self.observations_per_action.values() if n >= threshold)
        return trusted / len(self.observations_per_action)

    def has_class_coverage(self, object_class: int) -> bool:
        return any(getattr(r, "object_class", None) == object_class for r in self.rules)

# ============================================================
# MODULE: click_quantizer.py
# ============================================================






GRID_MAX_X = 63
GRID_MAX_Y = 63


@dataclass(frozen=True)
class ClickCandidate:
    x: int
    y: int
    rationale: str    # priors-only rationale string (no game-specific text)
    source: str       # "centroid" | "bbox_corner" | "edge_midpoint" | "quadrant_fallback"


def _clip(v: int, lo: int = 0, hi: int = GRID_MAX_X) -> int:
    return max(lo, min(hi, v))


def click_candidates(scene: SceneObservation, max_candidates: int = 20
                     ) -> list[ClickCandidate]:
    """Generate the priors-only ACTION6 candidate set for the given scene.

    Order: largest-object centroid first, then descending area for the
    remaining objects, then bbox corners of the top-3, then 9 quadrant
    fallbacks. Deduplicated by (x, y).
    """
    seen: set[tuple[int, int]] = set()
    out: list[ClickCandidate] = []

    # 1. Centroids of all detected objects (already area-sorted descending).
    for i, obj in enumerate(scene.objects):
        cr, cc = obj.centroid
        x, y = _clip(int(round(cc))), _clip(int(round(cr)))
        key = (x, y)
        if key in seen:
            continue
        seen.add(key)
        out.append(ClickCandidate(
            x=x, y=y,
            rationale=f"centroid of object rank {i} (color={obj.color}, area={obj.area})",
            source="centroid",
        ))
        if len(out) >= max_candidates:
            return out

    # 2. BBox corners of the top-3 objects — captures grab handles and
    #    "destination" cells common in object-puzzle games.
    for i, obj in enumerate(scene.objects[:3]):
        r0, c0, r1, c1 = obj.bbox
        for (yy, xx, corner) in [(r0, c0, "TL"), (r0, c1, "TR"),
                                  (r1, c0, "BL"), (r1, c1, "BR")]:
            x, y = _clip(int(xx)), _clip(int(yy))
            key = (x, y)
            if key in seen:
                continue
            seen.add(key)
            out.append(ClickCandidate(
                x=x, y=y,
                rationale=f"bbox {corner} of object rank {i}",
                source="bbox_corner",
            ))
            if len(out) >= max_candidates:
                return out

    # 3. Edge midpoints — captures "edge target" patterns the perceptor
    #    flagged via touches_edge but the centroid missed.
    for i, obj in enumerate(scene.objects[:3]):
        if not obj.touches_edge:
            continue
        r0, c0, r1, c1 = obj.bbox
        for (yy, xx, side) in [
            (r0, (c0 + c1) // 2, "top"),
            (r1, (c0 + c1) // 2, "bottom"),
            ((r0 + r1) // 2, c0, "left"),
            ((r0 + r1) // 2, c1, "right"),
        ]:
            x, y = _clip(int(xx)), _clip(int(yy))
            key = (x, y)
            if key in seen:
                continue
            seen.add(key)
            out.append(ClickCandidate(
                x=x, y=y,
                rationale=f"edge midpoint ({side}) of edge-touching object rank {i}",
                source="edge_midpoint",
            ))
            if len(out) >= max_candidates:
                return out

    # 4. 9-quadrant geometric fallback — generic coverage when objectness
    #    is empty (e.g. blank-grid start state). Pure geometry prior.
    if scene.rows > 0 and scene.cols > 0:
        max_r = scene.rows - 1
        max_c = scene.cols - 1
        grid_pts = [
            (max_r * a // 2, max_c * b // 2)
            for a in (0, 1, 2)
            for b in (0, 1, 2)
        ]
        for (yy, xx) in grid_pts:
            x, y = _clip(int(xx)), _clip(int(yy))
            key = (x, y)
            if key in seen:
                continue
            seen.add(key)
            out.append(ClickCandidate(
                x=x, y=y,
                rationale="9-quadrant geometric fallback",
                source="quadrant_fallback",
            ))
            if len(out) >= max_candidates:
                return out

    return out


def best_click_candidate(scene: SceneObservation,
                         policy_seeds_xy: Optional[list[tuple[int, int]]] = None
                         ) -> ClickCandidate:
    """Pick the single best candidate. Bias toward seed-aligned candidates
    if the resonance library provided any (x, y) hints from prior winning
    policies, else return the top of `click_candidates`.
    """
    candidates = click_candidates(scene)
    if not candidates:
        # Truly empty — center the grid.
        cx = (scene.cols - 1) // 2 if scene.cols else 0
        cy = (scene.rows - 1) // 2 if scene.rows else 0
        return ClickCandidate(
            x=_clip(cx), y=_clip(cy),
            rationale="empty scene; center-of-grid fallback",
            source="empty_fallback",
        )

    if policy_seeds_xy:
        # Pick the candidate closest to ANY seed coordinate.
        best = candidates[0]
        best_dist = float("inf")
        for c in candidates:
            for sx, sy in policy_seeds_xy:
                d = (c.x - sx) ** 2 + (c.y - sy) ** 2
                if d < best_dist:
                    best_dist = d
                    best = c
        return best

    return candidates[0]

# ============================================================
# MODULE: tracker_hungarian.py
# ============================================================







# Try scipy; fall back to greedy nearest-neighbor.
try:
    from scipy.optimize import linear_sum_assignment as _scipy_lsa
    _HAS_SCIPY = True
except ImportError:  # pragma: no cover - environment-dependent
    _scipy_lsa = None
    _HAS_SCIPY = False


# Gating: a match with cost above this is rejected (treated as birth+death).
# (a) DERIVED FROM PRIOR — under cohesion, an object that traveled more than
# half the grid in one step with no color persistence is more likely two
# distinct objects than a single tracked one. The gating constant is large
# enough to admit reasonable motion but small enough to reject identity
# collisions between obviously different objects.
GATING_COST = 50.0


@dataclass(frozen=True)
class TrackingCosts:
    alpha: float
    beta: float
    gamma: float


def _shape_hamming(a: Obj, b: Obj) -> float:
    """Per-cell binary-mask hamming distance, normalized by max bbox area.

    Each object is rasterized to its bbox, padded to the union size, and
    XORed. Returns a value in [0, 1].
    """
    ar0, ac0, ar1, ac1 = a.bbox
    br0, bc0, br1, bc1 = b.bbox
    ah, aw = ar1 - ar0 + 1, ac1 - ac0 + 1
    bh, bw = br1 - br0 + 1, bc1 - bc0 + 1
    h = max(ah, bh)
    w = max(aw, bw)
    mask_a = np.zeros((h, w), dtype=np.int8)
    mask_b = np.zeros((h, w), dtype=np.int8)
    # Center each mask in the union box for translation-invariant shape compare.
    off_a_r = (h - ah) // 2
    off_a_c = (w - aw) // 2
    off_b_r = (h - bh) // 2
    off_b_c = (w - bw) // 2
    mask_a[off_a_r:off_a_r + ah, off_a_c:off_a_c + aw] = 1
    mask_b[off_b_r:off_b_r + bh, off_b_c:off_b_c + bw] = 1
    diff = int(np.sum(mask_a ^ mask_b))
    return diff / float(h * w) if h * w else 0.0


def _centroid_dist(a: Obj, b: Obj) -> float:
    ar, ac = a.centroid
    br, bc = b.centroid
    return float(np.hypot(ar - br, ac - bc))


def _color_mismatch(a: Obj, b: Obj) -> float:
    return 0.0 if a.color == b.color else 1.0


def _pairwise_cost(prev: list[Obj], curr: list[Obj], w: TrackingCosts) -> np.ndarray:
    """Build the (len(prev), len(curr)) cost matrix."""
    m, n = len(prev), len(curr)
    if m == 0 or n == 0:
        return np.zeros((m, n), dtype=np.float64)
    cost = np.zeros((m, n), dtype=np.float64)
    for i, p in enumerate(prev):
        for j, c in enumerate(curr):
            cost[i, j] = (
                w.alpha * _centroid_dist(p, c)
                + w.beta * _shape_hamming(p, c)
                + w.gamma * _color_mismatch(p, c)
            )
    return cost


def _greedy_assign(cost: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    """Pure-numpy greedy nearest-neighbor fallback.

    Repeatedly pick the lowest-cost (i, j) pair, mark both used, repeat.
    Returns (row_ind, col_ind) in the same shape contract as scipy LSA
    (one array each, parallel indexing).

    Not optimal in general but tight for sparse correspondences with
    well-separated costs — which is the typical Spelke scene.
    """
    m, n = cost.shape
    if m == 0 or n == 0:
        return np.array([], dtype=np.intp), np.array([], dtype=np.intp)
    row_used = np.zeros(m, dtype=bool)
    col_used = np.zeros(n, dtype=bool)
    pairs: list[tuple[int, int]] = []
    # Sort all pairs by cost ascending, then greedy-fill.
    flat_idx = np.argsort(cost, axis=None, kind="stable")
    for fi in flat_idx:
        i, j = int(fi // n), int(fi % n)
        if row_used[i] or col_used[j]:
            continue
        pairs.append((i, j))
        row_used[i] = True
        col_used[j] = True
        if len(pairs) == min(m, n):
            break
    if not pairs:
        return np.array([], dtype=np.intp), np.array([], dtype=np.intp)
    ri = np.array([p[0] for p in pairs], dtype=np.intp)
    ci = np.array([p[1] for p in pairs], dtype=np.intp)
    return ri, ci


@dataclass
class HungarianTracker:
    """Stateless object-correspondence tracker between consecutive scenes.

    `costs` is read once at construction; if not supplied, pulled from
    CONFIG.tracker (frozen weights with provenance in config.py).
    """

    costs: Optional[TrackingCosts] = None
    gating_cost: float = GATING_COST
    # When True, prefer the greedy fallback even if scipy is available.
    # Useful for tests that exercise the fallback path on machines with scipy.
    force_greedy: bool = False

    def __post_init__(self) -> None:
        if self.costs is None:
            self.costs = TrackingCosts(
                alpha=CONFIG.tracker.alpha_centroid_dist,
                beta=CONFIG.tracker.beta_shape_hamming,
                gamma=CONFIG.tracker.gamma_color_mismatch,
            )

    def track(
        self,
        prev_scene: SceneObservation,
        curr_scene: SceneObservation,
    ) -> dict[int, Optional[int]]:
        """Match objects from prev_scene to curr_scene.

        Returns:
          {prev_obj_idx -> curr_obj_idx | None}
          A None value means the object was destroyed (no successor).
          Spawned objects (in curr but unmatched) are NOT in the dict;
          callers can derive them as `set(range(len(curr.objects))) - set(values)`.

        Idempotent: no mutation of inputs, no global state, deterministic.
        """
        prev_objs = list(prev_scene.objects)
        curr_objs = list(curr_scene.objects)

        # Edge cases.
        if not prev_objs:
            return {}
        if not curr_objs:
            return {i: None for i in range(len(prev_objs))}

        cost = _pairwise_cost(prev_objs, curr_objs, self.costs)

        # Choose solver.
        if _HAS_SCIPY and not self.force_greedy:
            # scipy returns assignment over min(m, n) pairs.
            row_ind, col_ind = _scipy_lsa(cost)
        else:
            row_ind, col_ind = _greedy_assign(cost)

        # Apply gating: reject matches whose cost is above threshold.
        result: dict[int, Optional[int]] = {i: None for i in range(len(prev_objs))}
        for ri, ci in zip(row_ind, col_ind):
            if cost[ri, ci] <= self.gating_cost:
                result[int(ri)] = int(ci)
            # else: leave as None (destroyed, even if scipy "matched" them)
        return result

    def spawned_indices(
        self,
        prev_scene: SceneObservation,
        curr_scene: SceneObservation,
        mapping: dict[int, Optional[int]],
    ) -> list[int]:
        """Indices in curr_scene.objects that were not matched (spawned)."""
        matched = {v for v in mapping.values() if v is not None}
        return [j for j in range(len(curr_scene.objects)) if j not in matched]

    def destroyed_indices(
        self,
        mapping: dict[int, Optional[int]],
    ) -> list[int]:
        """Indices in prev_scene.objects that have no successor."""
        return [i for i, v in mapping.items() if v is None]

# ============================================================
# MODULE: goal_inducer.py
# ============================================================






# Maximum free parameters per hypothesis — enforced at construction time.
# This is a Spelke-derived structural cap (the architect's must-fix #4):
# more than 3 params over the observed window admits curve-fitting, not induction.
MAX_FREE_PARAMS = 3


@dataclass(frozen=True)
class GoalHypothesis:
    """A single goal hypothesis with a ranked posterior score.

    `kind` selects the predicate family; `params` is a tuple of integer-valued
    parameters (≤3). `score` is the posterior-odds estimate from observed pairs.
    """
    kind: str
    params: tuple[int, ...]
    score: float
    support: int  # number of (Δlevels, scene-change) pairs that voted yes
    contradictions: int  # number of pairs that voted no

    def __post_init__(self) -> None:
        if len(self.params) > MAX_FREE_PARAMS:
            raise ValueError(
                f"hypothesis '{self.kind}' has {len(self.params)} params, "
                f"exceeds MAX_FREE_PARAMS={MAX_FREE_PARAMS}"
            )


@dataclass
class _ScenePair:
    """One (pre-scene, post-scene, Δlevels) observation."""
    pre: SceneObservation
    post: SceneObservation
    delta_levels: int


@dataclass
class GoalInducer:
    """Tracks scene-change pairs and emits ranked goal hypotheses.

    Pure Spelke priors: objectness (class counts), spatial coincidence
    (agent reached cell of class Y), numerosity (count == N).

    No knowledge of any specific game family.
    """

    pairs: list[_ScenePair] = field(default_factory=list)
    # Laplace pseudocount for posterior odds — a single Spelke prior over
    # "any predicate is plausible until contradicted".
    laplace_alpha: float = 1.0

    def observe(
        self,
        pre_scene: SceneObservation,
        post_scene: SceneObservation,
        delta_levels: int,
        scene_change_description: Optional[str] = None,
    ) -> None:
        """Record a (Δlevels, scene-change) pair.

        `scene_change_description` is accepted for API completeness (caller
        may have a human-readable diff) but is NOT used for induction — we
        derive predicates directly from the scene observations under priors.
        """
        # description is intentionally unused — induction works on the priors,
        # not on natural-language descriptions (Tier-1 STRICT: no LLM path).
        _ = scene_change_description
        self.pairs.append(_ScenePair(pre=pre_scene, post=post_scene,
                                      delta_levels=int(delta_levels)))

    # -- candidate enumeration ------------------------------------------------

    def _candidate_removed_all_classes(self) -> set[int]:
        """Classes that disappeared (count went to 0) in at least one pair."""
        out: set[int] = set()
        for p in self.pairs:
            pre_cnt = Counter(o.color for o in p.pre.objects)
            post_cnt = Counter(o.color for o in p.post.objects)
            for c in pre_cnt:
                if pre_cnt[c] > 0 and post_cnt.get(c, 0) == 0:
                    out.add(int(c))
        return out

    def _candidate_count_equals(self) -> set[tuple[int, int]]:
        """(class, N) pairs witnessed in any post-scene."""
        out: set[tuple[int, int]] = set()
        for p in self.pairs:
            post_cnt = Counter(o.color for o in p.post.objects)
            for c, n in post_cnt.items():
                out.add((int(c), int(n)))
            # also seed N=0 candidates from disappearance
            pre_cnt = Counter(o.color for o in p.pre.objects)
            for c in pre_cnt:
                if post_cnt.get(c, 0) == 0:
                    out.add((int(c), 0))
        return out

    def _candidate_agent_reached(self) -> set[tuple[int, int]]:
        """(agent_class, target_class) candidate pairs.

        Agent-class candidates: any color whose objects' centroids moved
        between pre and post (i.e. has non-zero translation).
        Target-class candidates: any color present in pre but with reduced
        count in post (consumed-on-contact pattern), OR any color whose cell
        is now occupied by the agent's class.

        We use cell-coincidence (post-grid value at agent's previous-or-
        adjacent position) as the spatial primitive.
        """
        out: set[tuple[int, int]] = set()
        for p in self.pairs:
            # Build per-class centroid sets pre and post.
            pre_by_color: dict[int, list[tuple[float, float]]] = {}
            post_by_color: dict[int, list[tuple[float, float]]] = {}
            for o in p.pre.objects:
                pre_by_color.setdefault(int(o.color), []).append(o.centroid)
            for o in p.post.objects:
                post_by_color.setdefault(int(o.color), []).append(o.centroid)

            # Agent class = a class whose mean centroid moved across the pair.
            moved_classes: set[int] = set()
            for c, pre_cents in pre_by_color.items():
                post_cents = post_by_color.get(c, [])
                if not post_cents or len(pre_cents) != len(post_cents):
                    continue
                pre_mean_r = sum(r for r, _ in pre_cents) / len(pre_cents)
                pre_mean_c = sum(cc for _, cc in pre_cents) / len(pre_cents)
                post_mean_r = sum(r for r, _ in post_cents) / len(post_cents)
                post_mean_c = sum(cc for _, cc in post_cents) / len(post_cents)
                if (abs(pre_mean_r - post_mean_r) + abs(pre_mean_c - post_mean_c)) > 0.5:
                    moved_classes.add(c)

            # Target class = a class whose count dropped (consumed on contact).
            shrunk_classes: set[int] = set()
            for c, pre_cents in pre_by_color.items():
                if len(post_by_color.get(c, [])) < len(pre_cents):
                    shrunk_classes.add(c)

            for ag in moved_classes:
                for tg in shrunk_classes:
                    if ag == tg:
                        continue
                    out.add((int(ag), int(tg)))
        return out

    # -- evaluation -----------------------------------------------------------

    def _evaluate_removed_all(self, cls: int) -> tuple[int, int]:
        """Return (support, contradictions) for 'all of class X removed → level'."""
        support = contradictions = 0
        for p in self.pairs:
            post_count = sum(1 for o in p.post.objects if o.color == cls)
            removed = (post_count == 0) and any(o.color == cls for o in p.pre.objects)
            if removed and p.delta_levels > 0:
                support += 1
            elif removed and p.delta_levels <= 0:
                contradictions += 1
            elif p.delta_levels > 0 and not removed:
                # level advanced without satisfying this hypothesis — contradicts
                contradictions += 1
        return support, contradictions

    def _evaluate_agent_reached(self, agent_cls: int, target_cls: int) -> tuple[int, int]:
        support = contradictions = 0
        for p in self.pairs:
            pre_target = sum(1 for o in p.pre.objects if o.color == target_cls)
            post_target = sum(1 for o in p.post.objects if o.color == target_cls)
            reached = (pre_target > 0) and (post_target < pre_target) and any(
                o.color == agent_cls for o in p.pre.objects
            )
            if reached and p.delta_levels > 0:
                support += 1
            elif reached and p.delta_levels <= 0:
                contradictions += 1
            elif p.delta_levels > 0 and not reached:
                contradictions += 1
        return support, contradictions

    def _evaluate_count_equals(self, cls: int, n: int) -> tuple[int, int]:
        support = contradictions = 0
        for p in self.pairs:
            post_count = sum(1 for o in p.post.objects if o.color == cls)
            holds = (post_count == n)
            if holds and p.delta_levels > 0:
                support += 1
            elif holds and p.delta_levels <= 0:
                contradictions += 1
            elif p.delta_levels > 0 and not holds:
                contradictions += 1
        return support, contradictions

    def _posterior(self, support: int, contradictions: int) -> float:
        """Laplace-smoothed posterior odds — a Spelke prior over predicate plausibility."""
        a = self.laplace_alpha
        return (support + a) / (support + contradictions + 2.0 * a)

    # -- public API -----------------------------------------------------------

    def hypothesize(self, top_k: int = 8) -> list[GoalHypothesis]:
        """Return ranked goal hypotheses (highest posterior first).

        Empty input → empty list. A hypothesis with zero support AND zero
        contradiction is omitted (no evidence either way).
        """
        if not self.pairs:
            return []

        hyps: list[GoalHypothesis] = []

        # Family 1: "all of class X removed" (1 free param)
        for cls in self._candidate_removed_all_classes():
            sup, con = self._evaluate_removed_all(cls)
            if sup == 0 and con == 0:
                continue
            hyps.append(GoalHypothesis(
                kind="removed_all_of_class",
                params=(int(cls),),
                score=self._posterior(sup, con),
                support=sup,
                contradictions=con,
            ))

        # Family 2: "agent reached cell of class Y" (2 free params)
        for agent_cls, target_cls in self._candidate_agent_reached():
            sup, con = self._evaluate_agent_reached(agent_cls, target_cls)
            if sup == 0 and con == 0:
                continue
            hyps.append(GoalHypothesis(
                kind="agent_reached_class",
                params=(int(agent_cls), int(target_cls)),
                score=self._posterior(sup, con),
                support=sup,
                contradictions=con,
            ))

        # Family 3: "count of class Z equals N" (2 free params)
        for cls, n in self._candidate_count_equals():
            sup, con = self._evaluate_count_equals(cls, n)
            if sup == 0 and con == 0:
                continue
            hyps.append(GoalHypothesis(
                kind="count_of_class_equals_N",
                params=(int(cls), int(n)),
                score=self._posterior(sup, con),
                support=sup,
                contradictions=con,
            ))

        # Rank: highest posterior score, then highest support, then lowest
        # contradiction count as a tiebreaker. Stable sort preserves insertion
        # order among ties for reproducibility.
        hyps.sort(key=lambda h: (-h.score, -h.support, h.contradictions, h.kind, h.params))
        return hyps[:top_k]

# ============================================================
# MODULE: abstain_policy.py
# ============================================================







# Spelke-derived constant: the score quadratic's marginal-half-life point
# is N = 2*h. This is geometric truth from the scoring rule, not a tunable.
HUMAN_BASELINE_MULTIPLIER = 2


@dataclass
class AbstainPolicy:
    """Decide when continuing on the current game is negative expected value.

    Stateless across instances; per-call state is read from the supplied
    tracker and world model. Construct fresh per game or reuse safely.
    """

    # Window size for novelty-plateau check. (a) DERIVED FROM PRIOR — the
    # smallest window that distinguishes plateau from short-term oscillation
    # under the Spelke continuity prior is K=5 (one Markov chain step plus
    # 4-of-5 corroboration). Documented here, NOT a free tunable.
    plateau_window_k: int = 5

    # (a) DERIVED FROM PRIOR — fingerprint-delta norm below this is "no
    # meaningful change". The scale is dimensionless because fingerprints
    # are L2-normalized; 0.01 is one percent of the unit ball.
    plateau_delta_threshold: float = 0.01

    # (b) BUDGET HEURISTIC — only relevant when no human baseline is known.
    # Pulled lazily from CONFIG so callers can override CONFIG in tests
    # without re-instantiating AbstainPolicy.
    _floor_min_actions: Optional[int] = None

    # Estimated human baseline for the current game. Optional — when None we
    # fall back to the config floor. The agent's outer loop may supply this
    # from the resonance library (prior winning policies) when available.
    estimated_human_baseline: Optional[int] = None

    # Recent fingerprint history (last K). Caller pushes to this each step;
    # we keep AbstainPolicy free of side-effects on tracker.
    fingerprint_history: list[np.ndarray] = field(default_factory=list)

    @property
    def min_actions(self) -> int:
        """Derived floor on action_counter before abstain may fire.

        Math:
          Δscore at N actions ≈ 2 h^2 / N^3
          Half-life point: N = 2h  (per docstring derivation)
        Floor: never below the config's min_actions_before_abstain to keep
        us from abstaining on trivial games where exploration is cheap.
        """
        floor = (
            self._floor_min_actions
            if self._floor_min_actions is not None
            else CONFIG.abstain.min_actions_before_abstain
        )
        if self.estimated_human_baseline is not None:
            derived = HUMAN_BASELINE_MULTIPLIER * int(self.estimated_human_baseline)
            return max(floor, derived)
        return floor

    def push_fingerprint(self, fp: np.ndarray) -> None:
        """Record one step's fingerprint for plateau detection.

        Keeps only the last plateau_window_k+1 vectors (we need K deltas).
        """
        self.fingerprint_history.append(np.asarray(fp, dtype=np.float32))
        cap = self.plateau_window_k + 1
        if len(self.fingerprint_history) > cap:
            self.fingerprint_history = self.fingerprint_history[-cap:]

    def _novelty_plateau(self) -> bool:
        """True if the last K fingerprint deltas are all below threshold.

        Returns False until we have K+1 fingerprints (insufficient evidence).
        """
        if len(self.fingerprint_history) < self.plateau_window_k + 1:
            return False
        recent = self.fingerprint_history[-(self.plateau_window_k + 1):]
        deltas = []
        for i in range(1, len(recent)):
            d = float(np.linalg.norm(recent[i] - recent[i - 1]))
            deltas.append(d)
        # All deltas below threshold → no novelty surface left to explore.
        return all(d < self.plateau_delta_threshold for d in deltas)

    def _world_model_variance(
        self,
        tracker: EpisodeTracker,
        world_model: WorldModel,
    ) -> float:
        """Predict-vs-observe disagreement rate over recent action records.

        For each closed ActionRecord (with post_levels_completed set), we
        compare the world model's predicted cells_changed against observed.
        Disagreement = predicted no-change but observed change, OR vice versa.

        Returns fraction in [0, 1]. 0 = perfect, 1 = always wrong.
        """
        # We need scene pairs aligned with action records. The tracker stores
        # scene N+1 alongside action record N (after observe()).
        closed_records = [
            (i, r) for i, r in enumerate(tracker.action_history)
            if r.post_levels_completed is not None and r.cells_changed is not None
        ]
        if len(closed_records) < CONFIG.world_model.min_observations_for_trust:
            # Not enough closed records to trust a variance estimate.
            return 0.0

        disagreements = 0
        total = 0
        # action record index i pairs with scenes[i] (pre) and scenes[i+1] (post)
        for i, rec in closed_records:
            if i + 1 >= len(tracker.scenes):
                continue
            pre_scene = tracker.scenes[i]
            try:
                predicted, conf = world_model.predict(pre_scene.grid, rec.action_name)
            except Exception:
                continue
            if conf <= 0.0:
                # WM doesn't have a confident prediction for this action yet —
                # exclude from variance (we'd otherwise penalize uncertainty
                # as if it were disagreement, which conflates two failure modes).
                continue
            predicted_change = not np.array_equal(predicted, pre_scene.grid)
            observed_change = (rec.cells_changed or 0) > 0
            if predicted_change != observed_change:
                disagreements += 1
            total += 1

        if total == 0:
            return 0.0
        return disagreements / total

    def should_abstain(
        self,
        tracker: EpisodeTracker,
        world_model: WorldModel,
    ) -> bool:
        """Three-conjunction abstain trigger.

        All three must hold:
          1. action_counter > min_actions  (quadratic-math floor)
          2. novelty plateau over last K actions
          3. world-model variance > threshold

        If any one is false, continue. This conservatism is deliberate —
        false-positive abstain throws away all remaining score on the level.
        """
        action_counter = len(tracker.action_history)
        if action_counter <= self.min_actions:
            return False
        if not self._novelty_plateau():
            return False
        var = self._world_model_variance(tracker, world_model)
        if var <= CONFIG.abstain.world_model_variance_threshold:
            return False
        return True

    def reason(
        self,
        tracker: EpisodeTracker,
        world_model: WorldModel,
    ) -> str:
        """Human-readable explanation of the current abstain decision.

        Returned even when should_abstain is False — names which condition
        held back the trigger. Used by tracker.set_rationale upstream.
        """
        action_counter = len(tracker.action_history)
        if action_counter <= self.min_actions:
            return (f"continue: action_counter={action_counter} <= "
                    f"min_actions={self.min_actions} (quadratic floor)")
        if not self._novelty_plateau():
            return (f"continue: novelty still active "
                    f"(K={self.plateau_window_k}, "
                    f"have {len(self.fingerprint_history)} fingerprints)")
        var = self._world_model_variance(tracker, world_model)
        if var <= CONFIG.abstain.world_model_variance_threshold:
            return (f"continue: world-model variance {var:.2f} "
                    f"<= threshold {CONFIG.abstain.world_model_variance_threshold:.2f}")
        return (f"abstain: actions={action_counter}, plateau=yes, "
                f"wm_variance={var:.2f}")

# ============================================================
# MODULE: action_search.py
# ============================================================



from typing import Any, Optional, Sequence


from arcengine import GameAction  # type: ignore[import-not-found]



def _name_of(action: Any) -> str:
    return getattr(action, "name", str(action))


def _action_dud_score(tracker: EpisodeTracker, name: str) -> float:
    """0.0 = no evidence of duddiness, 1.0 = certain dud (no effect ever)."""
    bucket = tracker.transition_signals.get(name)
    if not bucket or bucket["total"] == 0:
        return 0.0
    if bucket["cells_changed_sum"] == 0 and bucket["level_advances"] == 0:
        return 1.0
    return 0.0


def _action_advance_rate(tracker: EpisodeTracker, name: str) -> float:
    bucket = tracker.transition_signals.get(name)
    if not bucket or bucket["total"] == 0:
        return 0.0
    return bucket["level_advances"] / bucket["total"]


def _world_model_lookahead_score(
    world_model: Optional[WorldModel],
    grid: np.ndarray,
    action_name: str,
    tracker: EpisodeTracker,
) -> tuple[float, float]:
    """Return (score_bonus, confidence) for one-step lookahead under WM.

    Score bonus components (all priors-only):
      + 0.5 if the predicted grid differs from current (action is not a no-op)
      + 0.3 if predicted cells_changed > 0 AND action has zero observed advance
        (novel transition — worth probing under exploration)
      + 0.2 if the action has been seen before at high confidence and
        succeeded (cells_changed > 0 in prior observations)
    """
    if world_model is None:
        return 0.0, 0.0
    try:
        predicted, conf = world_model.predict(grid, action_name)
    except Exception:
        return 0.0, 0.0
    if conf <= 0.0:
        return 0.0, 0.0
    bonus = 0.0
    if not np.array_equal(predicted, grid):
        bonus += 0.5
        bucket = tracker.transition_signals.get(action_name)
        if not bucket or bucket["level_advances"] == 0:
            bonus += 0.3
        elif bucket["cells_changed_sum"] > 0:
            bonus += 0.2
    return bonus, conf


def select_action(
    scene: SceneObservation,
    tracker: EpisodeTracker,
    available_actions: Sequence[Any],
    policy_seeds: Sequence[Sequence[dict]],
    action_budget_remaining: int,
    world_model: Optional[WorldModel] = None,
) -> GameAction:
    """Return a GameAction from `available_actions` under Tier-1 priors only."""

    if not available_actions:
        tracker.record_action("RESET", int(GameAction.RESET.value), {},
                              pre_levels_completed=0)
        tracker.set_rationale("no available_actions reported; resetting")
        return GameAction.RESET

    # Step index → resonance seed votes.
    step_idx = len(tracker.action_history)
    seed_votes: dict[str, int] = {}
    seed_xy_hints: list[tuple[int, int]] = []
    for policy in policy_seeds:
        if step_idx < len(policy):
            entry = policy[step_idx]
            n = entry.get("action_name", "")
            if n:
                seed_votes[n] = seed_votes.get(n, 0) + 1
            data = entry.get("data") or {}
            if "x" in data and "y" in data:
                seed_xy_hints.append((int(data["x"]), int(data["y"])))

    # Score each available action.
    scored: list[tuple[float, Any, float]] = []  # (weight, action, wm_conf)
    for action in available_actions:
        name = _name_of(action)
        if _action_dud_score(tracker, name) >= 1.0 and action_budget_remaining > 5:
            continue
        weight = 1.0
        weight += 2.0 * _action_advance_rate(tracker, name)
        weight += 1.5 * seed_votes.get(name, 0)
        wm_bonus, wm_conf = _world_model_lookahead_score(
            world_model, scene.grid, name, tracker
        )
        weight += wm_bonus
        scored.append((weight, action, wm_conf))

    if not scored:
        action = random.choice(list(available_actions))
        chosen_name = _name_of(action)
        rationale = "all-actions-known-dud; random tiebreak"
        wm_conf_chosen = 0.0
    else:
        max_w = max(w for w, _, _ in scored)
        top = [(a, c) for w, a, c in scored if w == max_w]
        action, wm_conf_chosen = random.choice(top)
        chosen_name = _name_of(action)
        rationale = (
            f"priors+wm pick: advance={_action_advance_rate(tracker, chosen_name):.2f}, "
            f"seed_votes={seed_votes.get(chosen_name, 0)}, "
            f"wm_conf={wm_conf_chosen:.2f}"
        )

    # ACTION6 — quantize click coordinate via objectness prior, biased by
    # seed (x,y) hints if any.
    data: dict = {}
    if hasattr(action, "is_complex") and action.is_complex():
        cand = best_click_candidate(scene, policy_seeds_xy=seed_xy_hints or None)
        data = {"x": cand.x, "y": cand.y}
        action.set_data(data)
        rationale += f"; click=({cand.x},{cand.y}) {cand.source} [{cand.rationale}]"

    pre_lv = (
        tracker.action_history[-1].post_levels_completed
        if tracker.action_history
        and tracker.action_history[-1].post_levels_completed is not None
        else 0
    )
    tracker.record_action(chosen_name, int(getattr(action, "value", 0)), data,
                          pre_levels_completed=pre_lv)
    tracker.set_rationale(rationale)
    return action

# ============================================================
# MODULE: mcts_puct.py
# ============================================================







# ---------------------------------------------------------------------------
# Action handle — the only object the search ever mutates.
# ---------------------------------------------------------------------------

@dataclass
class ActionHandle:
    """Immutable-ish wrapper around a candidate action for the MCTS tree.

    Each handle owns a deep-copied `data` dict so sibling tree branches
    cannot leak click coordinates into each other. The `enum_ref` is kept
    only so the caller can re-bind data onto the canonical enum at the
    very end of `plan()`.
    """
    action_id: int               # the GameAction enum value
    action_name: str             # "ACTION1", "ACTION6", "RESET", ...
    is_complex: bool             # True for ACTION6 (needs x, y data)
    data: dict                   # always a fresh dict, deep-copied
    enum_ref: Any = None         # the original enum member (do not mutate)

    def key(self) -> tuple[str, tuple]:
        """Hashable identity for tree-edge bookkeeping."""
        if self.is_complex:
            xy = (self.data.get("x", -1), self.data.get("y", -1))
        else:
            xy = ()
        return (self.action_name, xy)


def make_handle_from_enum(enum_action: Any, data: Optional[dict] = None) -> ActionHandle:
    """Build a deep-copy-safe handle from a GameAction enum member.

    `data` is deep-copied; if not provided and the action is complex,
    we default to {}.
    """
    name = getattr(enum_action, "name", str(enum_action))
    aid = int(getattr(enum_action, "value", 0))
    is_complex = bool(getattr(enum_action, "is_complex", lambda: False)())
    return ActionHandle(
        action_id=aid,
        action_name=name,
        is_complex=is_complex,
        data=copy.deepcopy(data) if data else {},
        enum_ref=enum_action,
    )


# ---------------------------------------------------------------------------
# Tree node.
# ---------------------------------------------------------------------------

@dataclass
class _Node:
    """A node in the search tree. One node per (parent, action) edge."""
    grid_fingerprint: bytes      # hash of the grid this node represents
    depth: int
    parent: Optional["_Node"] = None
    incoming: Optional[ActionHandle] = None    # action that produced this node

    # Children, keyed by ActionHandle.key()
    children: dict[tuple, "_Node"] = field(default_factory=dict)
    child_handles: dict[tuple, ActionHandle] = field(default_factory=dict)
    child_priors: dict[tuple, float] = field(default_factory=dict)

    # Per-edge statistics — N(s,a), W(s,a), Q(s,a)
    n_sa: dict[tuple, int] = field(default_factory=dict)
    w_sa: dict[tuple, float] = field(default_factory=dict)

    # Node statistics
    n_s: int = 0                 # total visit count at this node
    is_terminal: bool = False    # set when WIN proxy fires or depth cap hit
    is_expanded: bool = False    # children populated?

    def q(self, edge_key: tuple) -> float:
        n = self.n_sa.get(edge_key, 0)
        if n == 0:
            return 0.0
        return self.w_sa.get(edge_key, 0.0) / n


# ---------------------------------------------------------------------------
# Planner.
# ---------------------------------------------------------------------------

@dataclass
class PlanResult:
    """Output of `plan()`."""
    chosen: ActionHandle
    root_stats: dict             # {edge_key_str: {"N":..,"Q":..,"P":..,"name":..}}
    rollouts_run: int
    wallclock_ms: float
    timed_out: bool


class MCTSPUCT:
    """PUCT planner with progressive widening and deep-copy-safe expansion.

    Construction is cheap — the costly work is in `plan(...)`.

    Constructor parameters:
      world_model_predict:
          callable(grid: np.ndarray, action_name: str) -> (next_grid, conf).
          Use `WorldModel.predict` in production. Tests pass synthetic stubs.

      click_candidates_fn:
          callable(scene_like) -> list of objects with .x, .y, .source attrs.
          Use `click_quantizer.click_candidates` in production. Tests stub it.

      last_known_progress_path:
          list[str] of action names from a winning resonance seed at this
          step_index, used to bias the prior. May be empty.

      historical_advance_rate:
          callable(action_name: str) -> float in [0, 1]. The episode-level
          empirical rate at which this action advanced a level. Drives the
          WIN proxy. In production, derived from EpisodeTracker; in tests,
          stubbed to a dict-lookup.

      rng:
          numpy.random.Generator for reproducible tie-breaking.

    Configuration (all from `CONFIG.mcts`):
      c_puct, max_depth, rollouts_per_action, hard_timeout_ms.
    """

    PROGRESSIVE_WIDENING_ALPHA = 0.5
    PROGRESSIVE_WIDENING_MIN = 3      # always expand at least this many click candidates

    # Reward constants (frozen per the spec)
    REWARD_WIN = 10.0
    REWARD_PER_ACTION = -0.01
    REWARD_NOVEL_FINGERPRINT = 0.10
    WIN_PROXY_ADVANCE_RATE_THRESHOLD = 0.5

    def __init__(
        self,
        world_model_predict: Callable[[np.ndarray, str], tuple[np.ndarray, float]],
        click_candidates_fn: Callable[[Any], Sequence[Any]],
        last_known_progress_path: Sequence[str] = (),
        historical_advance_rate: Optional[Callable[[str], float]] = None,
        rng: Optional[np.random.Generator] = None,
    ) -> None:
        self.world_model_predict = world_model_predict
        self.click_candidates_fn = click_candidates_fn
        self.progress_path = list(last_known_progress_path)
        self.advance_rate = historical_advance_rate or (lambda _name: 0.0)
        self.rng = rng or np.random.default_rng(0xA70AEE05)

        self.c_puct = CONFIG.mcts.c_puct
        self.max_depth = CONFIG.mcts.max_depth
        self.rollouts_target = CONFIG.mcts.rollouts_per_action
        self.timeout_ms = CONFIG.mcts.hard_timeout_ms

    # ------------------------------------------------------------------
    # Public API.
    # ------------------------------------------------------------------

    def plan(
        self,
        scene: Any,                      # SceneObservation-like, has .grid
        available_actions: Sequence[Any] # iterable of GameAction enum members
    ) -> PlanResult:
        """Run MCTS-PUCT and return the chosen action handle plus stats.

        The chosen handle is safe to mutate — its `data` dict is its own
        deep copy. Caller may apply `chosen.enum_ref.set_data(chosen.data)`
        exactly once to attach the click coordinates to the canonical enum
        member just before submitting to the engine.
        """
        t0 = time.perf_counter()
        deadline_s = t0 + (self.timeout_ms / 1000.0)

        root_grid = np.asarray(scene.grid).copy()
        root = _Node(
            grid_fingerprint=self._fp(root_grid),
            depth=0,
        )
        self._expand(root, root_grid, scene, available_actions)

        # Edge case: no actions to expand. Return a synthesized RESET handle
        # if it's in the available set, else first available, else None.
        if not root.children:
            return self._empty_plan(available_actions, t0)

        rollouts = 0
        timed_out = False
        for _ in range(self.rollouts_target):
            if time.perf_counter() >= deadline_s:
                timed_out = True
                break
            self._simulate(root, root_grid, scene, available_actions)
            rollouts += 1

        # Pick the most-visited child at root (standard AlphaZero choice).
        best_key = self._best_root_edge(root)
        chosen = root.child_handles[best_key]

        # Defensively re-deep-copy on the way out so the caller cannot
        # accidentally back-mutate any internal tree state.
        chosen_out = ActionHandle(
            action_id=chosen.action_id,
            action_name=chosen.action_name,
            is_complex=chosen.is_complex,
            data=copy.deepcopy(chosen.data),
            enum_ref=chosen.enum_ref,
        )

        wallclock_ms = (time.perf_counter() - t0) * 1000.0
        return PlanResult(
            chosen=chosen_out,
            root_stats=self._root_stats(root),
            rollouts_run=rollouts,
            wallclock_ms=wallclock_ms,
            timed_out=timed_out,
        )

    # ------------------------------------------------------------------
    # Tree expansion.
    # ------------------------------------------------------------------

    def _expand(
        self,
        node: _Node,
        grid: np.ndarray,
        scene: Any,
        available_actions: Sequence[Any],
    ) -> None:
        """Populate this node's children + priors. Idempotent."""
        if node.is_expanded:
            return
        node.is_expanded = True

        handles = self._enumerate_handles(node, scene, available_actions)
        if not handles:
            return

        # Compute unnormalized priors per handle, then normalize.
        raw: list[tuple[ActionHandle, float]] = []
        for h in handles:
            p = 1.0 if h.action_name in self.progress_path else 0.5
            raw.append((h, p))
        total = sum(p for _, p in raw) or 1.0

        for h, p in raw:
            k = h.key()
            if k in node.children:
                continue  # progressive widening may try to re-add; skip
            node.child_handles[k] = h
            node.child_priors[k] = p / total
            # Placeholder child node — instantiated on first descent.
            node.children[k] = _Node(
                grid_fingerprint=b"",  # filled on first descent
                depth=node.depth + 1,
                parent=node,
                incoming=h,
            )
            node.n_sa[k] = 0
            node.w_sa[k] = 0.0

    def _enumerate_handles(
        self,
        node: _Node,
        scene: Any,
        available_actions: Sequence[Any],
    ) -> list[ActionHandle]:
        """Build the candidate ActionHandle list for `node`, applying
        progressive widening to ACTION6 click candidates.

        Each handle gets a freshly deep-copied `data` dict — this is the
        Lane-A safety guarantee.
        """
        handles: list[ActionHandle] = []

        # How many click candidates to expand for this node?
        # Progressive widening: ceil(N(s) ** alpha), floored at the min.
        widen_k = max(
            self.PROGRESSIVE_WIDENING_MIN,
            int(math.ceil((node.n_s + 1) ** self.PROGRESSIVE_WIDENING_ALPHA)),
        )

        for ea in available_actions:
            is_complex = bool(getattr(ea, "is_complex", lambda: False)())
            if not is_complex:
                handles.append(make_handle_from_enum(ea, data={}))
                continue

            # Complex action → ClickQuantizer candidates, widened.
            cands = list(self.click_candidates_fn(scene))[:widen_k]
            if not cands:
                # No quantized candidates — fall back to grid centre.
                rows = int(getattr(scene, "rows", 0)) or int(scene.grid.shape[0])
                cols = int(getattr(scene, "cols", 0)) or int(scene.grid.shape[1])
                handles.append(make_handle_from_enum(
                    ea, data={"x": cols // 2, "y": rows // 2}
                ))
                continue
            for c in cands:
                handles.append(make_handle_from_enum(
                    ea, data={"x": int(c.x), "y": int(c.y)}
                ))
        return handles

    # ------------------------------------------------------------------
    # Simulation.
    # ------------------------------------------------------------------

    def _simulate(
        self,
        root: _Node,
        root_grid: np.ndarray,
        root_scene: Any,
        available_actions: Sequence[Any],
    ) -> None:
        """Run a single PUCT rollout from the root to a leaf/terminal."""
        node = root
        grid = root_grid.copy()
        scene = root_scene
        path: list[tuple[_Node, tuple]] = []       # (parent_node, edge_key)
        visited_fps: set[bytes] = {self._fp(grid)}
        cumulative_reward = 0.0

        for _depth in range(self.max_depth):
            if not node.children:
                break
            edge_key = self._select_edge(node)
            handle = node.child_handles[edge_key]
            child = node.children[edge_key]
            path.append((node, edge_key))

            # Forward sim — uses world_model.predict; THIS DOES NOT COUNT
            # against the real-environment action budget.
            try:
                next_grid, conf = self.world_model_predict(grid, handle.action_name)
            except Exception:
                conf = 0.0
                next_grid = grid

            # Reward shaping per the spec.
            cumulative_reward += self.REWARD_PER_ACTION
            fp = self._fp(next_grid)
            if fp not in visited_fps:
                cumulative_reward += self.REWARD_NOVEL_FINGERPRINT
                visited_fps.add(fp)

            # WIN proxy: high historical advance-rate AND material grid change.
            grid_changed = not np.array_equal(next_grid, grid)
            if (self.advance_rate(handle.action_name) >= self.WIN_PROXY_ADVANCE_RATE_THRESHOLD
                    and grid_changed
                    and conf >= 0.5):
                cumulative_reward += self.REWARD_WIN
                child.is_terminal = True

            # Advance into child.
            grid = next_grid
            if not child.grid_fingerprint:
                child.grid_fingerprint = fp
            node = child

            if node.is_terminal:
                break

            # Expand if the child has not been visited before.
            if not node.is_expanded:
                # Build a lightweight scene-like view over the simulated grid
                # so click_candidates_fn can still operate. We do NOT re-run
                # the perceptor here — that'd be a fat dependency for the
                # planner. Instead reuse root_scene; ClickQuantizer's job is
                # to give plausible candidates given some scene; using the
                # root scene's objects is a conservative approximation
                # consistent with "no Tier-2 contamination" — we never invent
                # game-specific positions, only reuse the prior frame's
                # objectness derived from the real observation.
                self._expand(node, grid, root_scene, available_actions)
                # Leaf rollout: one-step "value bootstrap" — already folded
                # into cumulative_reward above via the per-action terms.
                break

        # Backpropagate the cumulative reward up the visited path.
        self._backprop(root, path, cumulative_reward)

    def _select_edge(self, node: _Node) -> tuple:
        """PUCT edge selection at `node`. Returns the child edge key."""
        sqrt_ns = math.sqrt(max(node.n_s, 1))
        best_key = None
        best_score = -math.inf
        # Iterate in a deterministic order (sorted by key) for reproducibility.
        for k in sorted(node.child_handles.keys()):
            n_sa = node.n_sa.get(k, 0)
            q = node.q(k)
            p = node.child_priors.get(k, 0.0)
            u = self.c_puct * p * sqrt_ns / (1 + n_sa)
            score = q + u
            # Tiny noise on ties for deterministic-but-unbiased breaking.
            if score > best_score:
                best_score = score
                best_key = k
        if best_key is None:
            best_key = next(iter(node.child_handles))
        return best_key

    def _backprop(
        self,
        root: _Node,
        path: list[tuple[_Node, tuple]],
        reward: float,
    ) -> None:
        # Root visit first.
        root.n_s += 1
        for parent, edge_key in path:
            parent.n_sa[edge_key] = parent.n_sa.get(edge_key, 0) + 1
            parent.w_sa[edge_key] = parent.w_sa.get(edge_key, 0.0) + reward
            child = parent.children[edge_key]
            child.n_s += 1

    # ------------------------------------------------------------------
    # Helpers.
    # ------------------------------------------------------------------

    @staticmethod
    def _fp(grid: np.ndarray) -> bytes:
        """Cheap content fingerprint for novelty tracking."""
        a = np.ascontiguousarray(grid).astype(np.int32, copy=False)
        return a.tobytes()

    def _best_root_edge(self, root: _Node) -> tuple:
        """Most-visited child at root, tie-broken by Q then by sorted key."""
        items = list(root.child_handles.keys())
        if not items:
            raise RuntimeError("MCTS root has no children")
        items.sort()  # deterministic key order
        best = items[0]
        best_n = root.n_sa.get(best, 0)
        best_q = root.q(best)
        for k in items[1:]:
            n = root.n_sa.get(k, 0)
            q = root.q(k)
            if (n, q) > (best_n, best_q):
                best, best_n, best_q = k, n, q
        return best

    def _root_stats(self, root: _Node) -> dict:
        out: dict = {}
        for k, h in root.child_handles.items():
            out[self._stat_key(k)] = {
                "name": h.action_name,
                "data": dict(h.data),
                "N": root.n_sa.get(k, 0),
                "Q": root.q(k),
                "P": root.child_priors.get(k, 0.0),
            }
        out["__node__"] = {"N_total": root.n_s, "n_children": len(root.children)}
        return out

    @staticmethod
    def _stat_key(edge_key: tuple) -> str:
        name, xy = edge_key
        if xy:
            return f"{name}@{xy[0]},{xy[1]}"
        return name

    def _empty_plan(
        self,
        available_actions: Sequence[Any],
        t0: float,
    ) -> PlanResult:
        """Degenerate case — return a safe handle without crashing."""
        if available_actions:
            fallback = make_handle_from_enum(available_actions[0], data={})
        else:
            fallback = ActionHandle(
                action_id=0,
                action_name="RESET",
                is_complex=False,
                data={},
                enum_ref=None,
            )
        return PlanResult(
            chosen=fallback,
            root_stats={"__node__": {"N_total": 0, "n_children": 0}},
            rollouts_run=0,
            wallclock_ms=(time.perf_counter() - t0) * 1000.0,
            timed_out=False,
        )

# ============================================================
# MODULE: misfit_agent.py
# ============================================================








# Coverage threshold above which we gate up to MCTS planning. Below this
# we keep the cheap priors-only path — MCTS rollouts are wasted compute
# until the world model can predict at least some actions confidently.
# (b) BUDGET HEURISTIC — 0.3 = "at least one in three observed action types
# has crossed the trust threshold". Below that, rollouts mostly hit
# conf=0.0 leaves and degenerate to random walks.
MCTS_GATE_COVERAGE_THRESHOLD = 0.30


class Misfit(Agent):
    """The misfit substrate agent.

    Per-frame loop (see Agent.main upstream):
        1. perceive_frame(latest_frame.frame) → SceneObservation
        2. tracker.observe(latest_frame, scene)
        3. world_model.fit_with_refinement(observed transitions) — HRM outer loop
        4. fingerprint_episode(tracker) → 50-dim signature
        5. abstain_policy.push_fingerprint(signature) — plateau tracking
        6. resonance.find_k_nearest(signature) → prior winning policies
        7. If world_model.coverage() >= 0.3:
              MCTS-PUCT planner → ActionHandle
           Else:
              select_action(priors + 1-step lookahead + click quantizer)
        8. goal_inducer.observe(prev_scene, curr_scene, Δlevels) — every step
    """

    # StochasticGoose-confirmed: server does NOT enforce per-game cap.
    # 8h55m wall-clock self-kill is the real budget owner.
    MAX_ACTIONS = float("inf")  # type: ignore[assignment]
    LIBRARY_PATH: Optional[str] = None  # None → use default_library_path()
    HARD_WALL_CLOCK_SECONDS = 8 * 3600 + 50 * 60   # 8h50m — judges' Kaggle-reality fix

    # Gate above which MCTS-PUCT activates. Exposed as a class attribute so
    # tests can override it deterministically.
    MCTS_COVERAGE_GATE: float = MCTS_GATE_COVERAGE_THRESHOLD

    def __init__(self, *args: Any, **kwargs: Any) -> None:
        super().__init__(*args, **kwargs)
        self.tracker = EpisodeTracker(game_id=self.game_id)
        path = self.LIBRARY_PATH or str(default_library_path())
        self.library = ResonanceLibrary.load_or_create(path)
        self.world_model = WorldModel()

        # Wave-4 substrate modules — instantiated up front so they accumulate
        # state across the full episode. MCTS is lazy (built on first use) so
        # tests that exercise only the priors-fallback path don't pay for it.
        self.goal_inducer = GoalInducer()
        self.abstain_policy = AbstainPolicy()
        self.tracker_hungarian = HungarianTracker()
        self.mcts: Optional[MCTSPUCT] = None  # built lazily in choose_action

        self._start_time = time.time()
        self._wm_observations: list[dict] = []

    @property
    def name(self) -> str:
        return f"{super().name}.{self.MAX_ACTIONS}"

    def _has_time_elapsed(self) -> bool:
        """Wall-clock self-kill — judges' Kaggle-reality must-fix."""
        return (time.time() - self._start_time) >= self.HARD_WALL_CLOCK_SECONDS

    def is_done(self, frames: list[FrameData], latest_frame: FrameData) -> bool:
        """Stop on WIN, on abstain trigger, or when wall-clock budget exhausted."""
        try:
            if latest_frame.state is GameState.WIN:
                return True
            # Abstain check — three-conjunction trigger (min actions + novelty
            # plateau + world-model variance). False-positive abstain throws
            # away score; the policy is deliberately conservative.
            if self.abstain_policy.should_abstain(self.tracker, self.world_model):
                return True
            if self._has_time_elapsed():
                return True
        except Exception:
            return True  # bail out on unexpected state — never block the framework
        return False

    def _ensure_mcts(self) -> MCTSPUCT:
        """Lazy-build the MCTS planner on first use.

        Wired against `self.world_model.predict`, `click_candidates`, an empty
        progress path (resonance seeds are still consumed by select_action;
        MCTS uses its own PUCT prior over expanded children), and an advance-
        rate callable backed by the episode tracker.
        """
        if self.mcts is None:
            def _advance_rate(name: str) -> float:
                bucket = self.tracker.transition_signals.get(name)
                if not bucket or bucket["total"] == 0:
                    return 0.0
                return bucket["level_advances"] / bucket["total"]

            self.mcts = MCTSPUCT(
                world_model_predict=self.world_model.predict,
                click_candidates_fn=click_candidates,
                last_known_progress_path=(),
                historical_advance_rate=_advance_rate,
            )
        return self.mcts

    def choose_action(
        self,
        frames: list[FrameData],
        latest_frame: FrameData,
    ) -> GameAction:
        try:
            # Game not started or game-over — reset.
            if latest_frame.state in (GameState.NOT_PLAYED, GameState.GAME_OVER):
                self.tracker.observe_terminal(latest_frame)
                action = GameAction.RESET
                action.reasoning = "game needs reset"
                return action

            # Perceive the current frame under Spelke priors.
            scene = perceive_frame(latest_frame.frame)
            self.tracker.observe(latest_frame, scene)

            # Refit the world model whenever a fresh transition just landed
            # (uses HRM outer refinement loop, not plain .fit()).
            self._maybe_refit_world_model()

            # Pull resonance seeds — prior winning policies for similar episodes.
            signature = fingerprint_episode(self.tracker)
            seeds = self.library.retrieve_policy_seeds(signature, k=5)

            # Push the per-step fingerprint to AbstainPolicy for plateau tracking.
            # The abstain check itself fires in is_done, not here — choose_action
            # only feeds the signal.
            self.abstain_policy.push_fingerprint(signature)

            # Feed GoalInducer with the latest (s, a, s', Δlevels) pair so the
            # ranked hypothesis list stays current. This is the Spelke-numerosity
            # path — we don't act on the hypothesis directly here (that arrives
            # in a later wave), but we accumulate it for the cleanup tag.
            self._observe_goal_pair(latest_frame, scene)

            # Resolve available actions. Per StochasticGoose: the gateway sends
            # raw ints [1,2,...,6,7] rather than GameAction enum members.
            raw_avail = getattr(latest_frame, "available_actions", None) or []
            available = []
            for a in raw_avail:
                aid = a.value if hasattr(a, "value") else int(a)
                try:
                    available.append(GameAction.from_id(aid))
                except Exception:
                    # If from_id is missing, try enum lookup by value.
                    for ga in GameAction:
                        if int(getattr(ga, "value", -1)) == aid:
                            available.append(ga)
                            break

            # ----------------------------------------------------------------
            # COVERAGE-GATED MCTS: only run the planner once the world model
            # can predict at least some actions confidently. Below the gate,
            # rollouts mostly hit conf=0.0 leaves and degenerate to random
            # walks — wasted compute. Above the gate, MCTS is the only path
            # to a quadratic score lift per the (h/N)^2 scoring rule.
            # ----------------------------------------------------------------
            coverage = self.world_model.coverage()
            if available and coverage >= self.MCTS_COVERAGE_GATE:
                action = self._choose_via_mcts(scene, available)
                # The tracker rationale and action record are populated by
                # _choose_via_mcts so we can fall through to the return.
                return action

            # Below coverage gate — fall back to priors + 1-step lookahead
            # + click quantizer (existing select_action path).
            action = select_action(
                scene=scene,
                tracker=self.tracker,
                available_actions=available,
                policy_seeds=seeds,
                action_budget_remaining=10_000,  # wall-clock bound, not action bound
                world_model=self.world_model,
            )

            # Attach the rationale the substrate produced (for traceability).
            rationale = self.tracker.last_action_rationale or "priors-only fallback"
            if action.is_simple():
                action.reasoning = rationale
            elif action.is_complex():
                action.reasoning = {
                    "desired_action": str(getattr(action, "value", "")),
                    "rationale": rationale,
                }
            return action
        except Exception as e:
            # Never crash the framework — defensive fallback.
            try:
                action = GameAction.ACTION1
                action.reasoning = f"fallback after exception: {type(e).__name__}: {e}"
                return action
            except Exception:
                return GameAction.RESET

    def _choose_via_mcts(
        self,
        scene: Any,
        available: list[Any],
    ) -> GameAction:
        """Run MCTS-PUCT and bind the chosen handle to a canonical enum.

        Important: ActionHandle's `data` dict is deep-copied (Lane-A safety).
        We apply `set_data(chosen.data)` to the canonical enum exactly ONCE
        here — the same contract proven in test_action6_mutation_safety.
        """
        mcts = self._ensure_mcts()
        plan = mcts.plan(scene=scene, available_actions=available)
        chosen: ActionHandle = plan.chosen

        # Bind the chosen handle's data onto the canonical enum member.
        # If enum_ref is None (empty-plan degenerate path), fall back to
        # first available action; on truly empty avail, reset.
        enum_action = chosen.enum_ref
        if enum_action is None:
            if available:
                enum_action = available[0]
            else:
                action = GameAction.RESET
                action.reasoning = "mcts empty-plan + no available actions; reset"
                self.tracker.record_action(
                    "RESET", int(GameAction.RESET.value), {},
                    pre_levels_completed=self._pre_lv(),
                )
                self.tracker.set_rationale("mcts empty-plan; reset")
                return action

        # Deep-copy the data dict one more time before set_data — the handle
        # already deep-copies internally, but a second copy at the bind site
        # is the cheap belt-and-braces guarantee for Lane-A.
        bound_data = copy.deepcopy(chosen.data) if chosen.data else {}
        if chosen.is_complex and bound_data:
            try:
                enum_action.set_data(bound_data)
            except Exception:
                # If the enum rejects the data shape for any reason, leave it
                # un-set rather than crash the framework.
                pass

        # Build rationale and tracker bookkeeping.
        rationale = (
            f"mcts pick: name={chosen.action_name} "
            f"rollouts={plan.rollouts_run} ms={plan.wallclock_ms:.1f} "
            f"timed_out={plan.timed_out} coverage={self.world_model.coverage():.2f}"
        )
        if chosen.is_complex and bound_data:
            rationale += f" data={bound_data}"

        self.tracker.record_action(
            chosen.action_name,
            int(getattr(enum_action, "value", chosen.action_id)),
            bound_data,
            pre_levels_completed=self._pre_lv(),
        )
        self.tracker.set_rationale(rationale)

        if hasattr(enum_action, "is_simple") and enum_action.is_simple():
            enum_action.reasoning = rationale
        else:
            enum_action.reasoning = {
                "desired_action": str(getattr(enum_action, "value", "")),
                "rationale": rationale,
            }
        return enum_action

    def _pre_lv(self) -> int:
        """Pre-action level count for tracker.record_action."""
        if (self.tracker.action_history
                and self.tracker.action_history[-1].post_levels_completed is not None):
            return self.tracker.action_history[-1].post_levels_completed
        return 0

    def _observe_goal_pair(self, latest_frame: Any, scene: Any) -> None:
        """Feed GoalInducer with one (pre, post, Δlevels) pair if we have one.

        Requires at least two scenes and a closed prior action record.
        """
        if len(self.tracker.scenes) < 2:
            return
        prev_scene = self.tracker.scenes[-2]
        curr_scene = self.tracker.scenes[-1]
        # Δlevels: use the most recently closed action record's level delta.
        delta_levels = 0
        if self.tracker.action_history:
            rec = self.tracker.action_history[-1]
            if rec.post_levels_completed is not None:
                delta_levels = int(rec.post_levels_completed - rec.pre_levels_completed)
        try:
            self.goal_inducer.observe(prev_scene, curr_scene, delta_levels)
        except Exception:
            # GoalInducer is fail-soft — bad input must not crash the loop.
            pass

    def _maybe_refit_world_model(self) -> None:
        """Build a (s,a,s') observation from the latest two scenes + last action
        and refit rule templates. Called once per step after self.tracker.observe.

        Uses `fit_with_refinement(max_iters=4)` — the HRM outer-loop variant —
        instead of plain `.fit()`. Per the arcprize.org HRM analysis blog
        (2025-08-15), refinement was +13pp from 0→1 and roughly doubled the
        gain from 1→8. We also enrich each per-class observation with the
        HungarianTracker correspondence dict so per-class rules see object
        identity, not just summary counts.
        """
        if len(self.tracker.scenes) < 2 or not self.tracker.action_history:
            return
        last_action = self.tracker.action_history[-1]
        prev_scene = self.tracker.scenes[-2]
        curr_scene = self.tracker.scenes[-1]
        if prev_scene.grid.shape != curr_scene.grid.shape:
            return

        # Compute correspondences once for this transition. Folded into each
        # per-class obs below so the rule fitter has access if it wants it.
        try:
            correspondences = observe_hungarian(prev_scene, curr_scene, self.tracker)
        except Exception:
            correspondences = None

        classes_involved = sorted(set(int(o.color) for o in prev_scene.objects)
                                  | set(int(o.color) for o in curr_scene.objects))

        # Group object summaries by color for the rule.fit consumers.
        def _grouped(scene):
            by_color: dict[int, list[dict]] = {}
            for o in scene.objects:
                by_color.setdefault(int(o.color), []).append({
                    "centroid": o.centroid,
                    "area": int(o.area),
                })
            return by_color

        prev_by_color = _grouped(prev_scene)
        curr_by_color = _grouped(curr_scene)

        # Emit one observation per class so the composer can fit per-class rules.
        for cls in classes_involved:
            obs = {
                "action_name": last_action.action_name,
                "classes_involved": [cls],
                "pre_objects_of_class": prev_by_color.get(cls, []),
                "post_objects_of_class": curr_by_color.get(cls, []),
            }
            if correspondences is not None:
                # Optional enrichment — does not change the existing
                # Translate/NoOp.fit contract (those keys are ignored if
                # unknown), but lets downstream rule families key on identity.
                obs["correspondences"] = correspondences
            self._wm_observations.append(obs)

        # Refit every 5 steps to bound cost — now via HRM outer loop.
        if len(self.tracker.action_history) % 5 == 0:
            try:
                self.world_model.fit_with_refinement(
                    self._wm_observations, max_iters=4,
                )
            except Exception:
                pass

    def cleanup(self, scorecard: Optional[Any] = None) -> None:
        """On win, persist the winning policy to the resonance library.

        Also persists the GoalInducer's top hypothesis as a tag on the library
        entry's game_id so future K-NN retrievals can correlate "what worked"
        with "what we believed the goal was at win time".
        """
        if self.frames and self.frames[-1].state is GameState.WIN:
            # Top goal hypothesis at win — tag for later analysis.
            top_hyp_tag = self._top_hypothesis_tag()

            # Library API accepts a `game_id`; we suffix the tag onto it so
            # we don't break the existing schema (which has no free-text tag
            # column). This is the smallest-surface change that preserves
            # both honesty and forward-compat with v2 schema.
            tagged_game_id = (
                f"{self.game_id}#{top_hyp_tag}" if top_hyp_tag else self.game_id
            )
            self.library.record_solved(
                fingerprint=fingerprint_episode(self.tracker),
                winning_policy=self.tracker.action_history.copy(),
                composite_score=self._compute_composite_score(),
                source="self-solved",
                game_id=tagged_game_id,
            )
            self.library.flush_to_disk()
        super().cleanup(scorecard)

    def _top_hypothesis_tag(self) -> str:
        """Best goal hypothesis at win, encoded as a short tag string.

        Returns empty string if the inducer has no scored hypothesis. The
        tag is `<kind>(<params>)`, e.g. `removed_all_of_class(2)` or
        `agent_reached_class(3,7)`. Underscores in `kind` are preserved.
        """
        try:
            top = self.goal_inducer.hypothesize(top_k=1)
        except Exception:
            return ""
        if not top:
            return ""
        h = top[0]
        params_str = ",".join(str(p) for p in h.params)
        return f"{h.kind}({params_str})"

    def _compute_composite_score(self) -> float:
        """Cheap proxy for the (human_baseline / ai_actions)^2 scoring rule.
        Without per-level human baselines on hand, we use a budget-fraction proxy.
        """
        if not self.action_counter:
            return 1.0
        budget_left = max(0, self.MAX_ACTIONS - self.action_counter)
        return budget_left / self.MAX_ACTIONS

# ============================================================
# Framework entry point — Misfit registered as MyAgent.
# ============================================================
MyAgent = Misfit


In [ ]:
# Cell 3 — KAGGLE_IS_COMPETITION_RERUN bootstrap + slim init rewrite.
#
# Detects rerun mode (Phase B = real eval). Phase A skips this whole
# block and falls through to Cell 4 which writes the dummy parquet.
import os
from pathlib import Path

if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    # Wait for the eval gateway to come up before doing anything else.
    !curl --fail --retry 999 --retry-all-errors --retry-delay 5 \
          --retry-max-time 600 http://gateway:8001/api/games

    # Copy the bundled framework to a writable location (hedged path).
    _FW_CANDIDATES = [
        '/kaggle/input/arc-prize-2026-arc-agi-3/ARC-AGI-3-Agents',
        '/kaggle/input/competitions/arc-prize-2026-arc-agi-3/ARC-AGI-3-Agents',
    ]
    _fw_src = next((p for p in _FW_CANDIDATES if os.path.isdir(p)), None)
    assert _fw_src, f'no ARC-AGI-3-Agents found at any of: {_FW_CANDIDATES}'
    !cp -r {_fw_src} /kaggle/working/ARC-AGI-3-Agents

    # Install our agent module into agents/templates/.
    !cp /kaggle/working/my_agent.py \
        /kaggle/working/ARC-AGI-3-Agents/agents/templates/my_agent.py

    # ------------------------------------------------------------------
    # SLIM agents/__init__.py — only Random + Misfit registered.
    # langgraph / smolagents / openai are NOT in the offline wheel set;
    # eagerly importing them crashes the framework load before MyAgent
    # gets registered. This rewrite mirrors the patch already applied
    # to _research/ARC-AGI-3-Agents/agents/__init__.py in the source repo.
    # ------------------------------------------------------------------
    _slim_init = '"""Slim init — only Random + Misfit, no langgraph/smolagents/openai deps.\n\nThis is the Kaggle-deploy rewrite. Mirrors the Day-12 architect plan\nslim version at _research/ARC-AGI-3-Agents/agents/__init__.py.\n"""\nfrom typing import Type, cast\nfrom dotenv import load_dotenv\nfrom .agent import Agent, Playback\nfrom .recorder import Recorder\nfrom .swarm import Swarm\nfrom .templates.random_agent import Random\nfrom .templates.my_agent import MyAgent\n\nload_dotenv()\n\n# Misfit is registered under BOTH `myagent` (canonical framework slug)\n# and `misfit` (operator-friendly alias). Random stays for sanity runs.\nAVAILABLE_AGENTS: dict[str, Type[Agent]] = {\n    "random": cast(Type[Agent], Random),\n    "myagent": cast(Type[Agent], MyAgent),\n    "misfit": cast(Type[Agent], MyAgent),\n}\n\n# add all recording files as valid agent names (Playback)\nfor rec in Recorder.list():\n    AVAILABLE_AGENTS[rec] = Playback\n\n__all__ = ["Swarm", "Random", "MyAgent", "Agent", "Recorder", "Playback", "AVAILABLE_AGENTS"]\n'
    Path('/kaggle/working/ARC-AGI-3-Agents/agents/__init__.py').write_text(
        _slim_init, encoding='utf-8'
    )

    # .env — gateway endpoint per the StochasticGoose rerun contract.
    _env_body = 'SCHEME=http\nHOST=gateway\nPORT=8001\nARC_API_KEY=test-key-123\nARC_BASE_URL=http://gateway:8001/\nOPERATION_MODE=online\nENVIRONMENTS_DIR=\nRECORDINGS_DIR=/kaggle/working/server_recording\n'
    Path('/kaggle/working/ARC-AGI-3-Agents/.env').write_text(
        _env_body, encoding='utf-8'
    )

    print('Phase B bootstrap complete — agent run trigger fires in Cell 4.')
else:
    print('Not in KAGGLE_IS_COMPETITION_RERUN; Phase A dummy parquet path in Cell 4.')


In [ ]:
# Cell 4 — Phase A dummy submission.parquet + Phase B agent run trigger.
#
# Phase A (Kaggle save-and-validate, KAGGLE_IS_COMPETITION_RERUN unset):
#   Just write the dummy parquet so Kaggle accepts the submission.
#
# Phase B (real rerun, KAGGLE_IS_COMPETITION_RERUN=1):
#   Actually run the agent against the gateway. The framework writes its
#   own submission.parquet via Recorder so we don't write a dummy here.
import os

if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    # Phase B — agent run trigger.
    !cd /kaggle/working/ARC-AGI-3-Agents && \
        MPLBACKEND=agg \
        python main.py --agent myagent
else:
    # Phase A — dummy parquet so Kaggle accepts the save.
    import pandas as pd
    pd.DataFrame(
        data=[['1_0', '1', True, 1]],
        columns=['row_id', 'game_id', 'end_of_game', 'score'],
    ).to_parquet('/kaggle/working/submission.parquet', index=False)
    print('Wrote dummy submission.parquet for Phase A validation.')
